# Imports

In [ ]:
!pip install playwright
!python -m playwright install
!pip install bs4
!pip install requests
!pip install yfinance
!pip install pandas_ta
!pip install TA-Lib
!playwright install-deps

Playwright Host validation warning: 
╔══════════════════════════════════════════════════════╗
║ Host system is missing dependencies to run browsers. ║
║ Please install them with the following command:      ║
║                                                      ║
║     playwright install-deps                          ║
║                                                      ║
║ Alternatively, use apt:                              ║
║     apt-get install libxcomposite1\                  ║
║         libgtk-3-0\                                  ║
║         libatk1.0-0                                  ║
║                                                      ║
║ <3 Playwright Team                                   ║
╚══════════════════════════════════════════════════════╝
    at validateDependenciesLinux (/usr/local/lib/python3.12/dist-packages/playwright/driver/package/lib/server/registry/dependencies.js:269:9)
    at process.processTicksAndRejections (node:internal/process/task_queues:103

In [ ]:
!playwright install-deps
# Fix dpkg to allow further installations
!sudo dpkg --configure -a

# Install Playwright browser dependencies after dpkg is fixed
!playwright install-deps

Installing dependencies...
Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,870 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,623 kB]
Fetched 12.5 MB in 4s (3,060 kB/s)
R

# Technical Analysis

In [ ]:
#step 1, Extract bullish stocks
#now just change URL and query if use want...
from playwright.async_api import async_playwright
import asyncio
import csv
from datetime import date, timedelta, datetime
import os

title = ""

async def capture_chartink_table_with_query(
    url,
    query_text,
    total_pages=3,
    output_csv="chartink_data.csv",
    viewport={"width": 1280, "height": 800},
    wait_for=2000
):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(viewport=viewport)

        page_content = await page.goto(url, wait_until="networkidle")
        print(page_content)
        await page.wait_for_timeout(wait_for)

        if query_text is not None:
            # 1️⃣ Type query in textarea
            textarea_selector = "textarea[name='comment']"
            await page.fill(textarea_selector, query_text)
            print(f"📝 Typed query: {query_text}")

            # 2️⃣ Click "Generate" button
            generate_btn_selector = "button:has-text('Generate')"
            await page.click(generate_btn_selector)
            print("➡️ Clicked Generate button")
            await page.wait_for_timeout(1000)

            # 3️⃣ Click "Run Scan" button
            run_scan_selector = "div[title='Click to run scan']"
            await page.click(run_scan_selector)
            print("▶️ Clicked Run Scan button")
            await page.wait_for_timeout(1000)

        # 4️⃣ Extract table data
        all_rows = []
        headers = []

        for i in range(1, total_pages + 1):
            print(f"\n📄 Extracting data from page {i}...")

            table_selector = "table.rounded-b-\[0\.4375rem\].min-w-max.md\:w-full.whitespace-nowrap"
            await page.wait_for_selector(table_selector)

            # Extract headers only on first page
            if i == 1:
                ths = await page.query_selector_all(f"{table_selector} thead th")
                headers = [await th.inner_text() for th in ths]
                all_rows.append(headers)

            # Extract rows
            rows = await page.query_selector_all(f"{table_selector} tbody tr")
            for row in rows:
                cells = await row.query_selector_all("td")
                row_data = [await cell.inner_text() for cell in cells]
                all_rows.append(row_data)

            print(f"✅ Page {i} -> {len(rows)} rows collected")

            # Navigate to next page if exists
            if i < total_pages:
                try:
                    next_page_num = str(i + 1)
                    pagination = page.locator("div.hidden.sm\\:flex.w-fit.items-center")
                    await pagination.locator(f'button:text("{next_page_num}")').click()
                    await page.wait_for_load_state("networkidle")
                    await page.wait_for_timeout(wait_for)
                    print(f"➡️ Clicked page {next_page_num}")
                except Exception as e:
                    print(f"⚠️ Could not navigate to page {i+1}: {e}")
                    break

        # Save to CSV
        with open(output_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerows(all_rows)

        print(f"\n💾 Data saved to {output_csv}")
        await browser.close()


# Example usage
async def main():
    url_macd_bullish="https://chartink.com/screener/macd-bullish-crossover"
    url_macd_bullish_plus_rsi="https://chartink.com/screener/rsi-crossed-above-60-macd-cross-over-macd-signal"
    url_macd_bullish_plus_adx="https://chartink.com/screener/macd-bullish-crossover-with-adx-25-adx-di-adx-di"

    #inputs
    url = url_macd_bullish
    specification =  None
    # specification = "RSI in between 60 to 65"
    query = specification

    #title.csv
    today = date.today()
    print("Today's date:", today)

    technique = None

    if url == url_macd_bullish:
        technique = "MACD_Bullish"
    elif url == url_macd_bullish_plus_rsi:
        technique = "MACD_Bullish_RSI"
    elif url == url_macd_bullish_plus_adx:
        technique = "MACD_Bullish_ADX"
    else:
        technique = "None"

    if query is not None:
        technique = technique + "_" + specification

    title = technique + ".csv"
    print(title)

    totla_pages = 3


    #folder and files
    today_folder = date.today().strftime("%d-%m-%Y")
    print(today_folder)

    # Create folder if it doesn't exist
    if not os.path.exists(today_folder):
        os.makedirs(today_folder)

    output_csv = os.path.join(today_folder, title)
    print(output_csv)

    await capture_chartink_table_with_query(
        url=url,
        query_text=query,
        total_pages=totla_pages,
        output_csv= output_csv
    )

    return title

title = await main()

Today's date: 2026-01-19
MACD_Bullish.csv
19-01-2026
19-01-2026/MACD_Bullish.csv


<>:52: SyntaxWarning: invalid escape sequence '\['
/tmp/ipython-input-3006558751.py:52: SyntaxWarning: invalid escape sequence '\['
  table_selector = "table.rounded-b-\[0\.4375rem\].min-w-max.md\:w-full.whitespace-nowrap"


<Response url='https://chartink.com/screener/macd-bullish-crossover' request=<Request url='https://chartink.com/screener/macd-bullish-crossover' method='GET'>>

📄 Extracting data from page 1...
✅ Page 1 -> 20 rows collected
➡️ Clicked page 2

📄 Extracting data from page 2...
✅ Page 2 -> 20 rows collected
➡️ Clicked page 3

📄 Extracting data from page 3...
✅ Page 3 -> 20 rows collected

💾 Data saved to 19-01-2026/MACD_Bullish.csv


In [ ]:
#step 2: trend and candle stick pattern forms?
#Trend giver
import yfinance as yf
import pandas_ta as ta
import pandas as pd
import talib
from dateutil.relativedelta import relativedelta
from datetime import date

def get_trend(ticker, start_date, end_date):

      data = yf.download(f"{ticker}.NS", start=start_date, end=end_date)

      # Flatten the MultiIndex columns
      data.columns = ['_'.join(col).strip() for col in data.columns.values]
      # print(data.columns)

      # Check for non-numeric values in the 'Close_INDOFARM.NS' column
      # print(data[f'Close_{ticker}.NS'][~pd.to_numeric(data[f'Close_{ticker}.NS'], errors='coerce').notna()])

      # Example: Check if trend is bullish using a moving average crossover
      data['SMA_20'] = ta.sma(data[f'Close_{ticker}.NS'], length=20)
      data['SMA_50'] = ta.sma(data[f'Close_{ticker}.NS'], length=50)

      # Drop rows with NaN values in SMA columns before calculating trend
      data.dropna(subset=['SMA_20', 'SMA_50'], inplace=True)

      curr_trend = ""
      if not data.empty and data['SMA_20'].iloc[-1] > data['SMA_50'].iloc[-1]:
          # print(f"📈 Current trend for {ticker}: Bullish")
          curr_trend = 1
      else:
          # print(f"📉 Current trend for {ticker}: Bearish or not enough data")
          curr_trend = 0 # Assign Bearish if no data or not bullish

      return curr_trend

today = date.today()
path = today.strftime("%d-%m-%Y") + "/" + title

end_date = date.today()
start_date = end_date - relativedelta(months=6)

try:
    df = pd.read_csv(path)
    print(f"Successfully loaded {title}")
except FileNotFoundError:
    print(f"Error: The file '{title}' was not found.")
except Exception as e:
    print(f"An error occurred while loading the CSV: {e}")

df['trend'] = df['Symbol'].apply(lambda x: get_trend(x, start_date, end_date))
print(df.head())


#candle stick pattern given
def get_pattern(ticker, start_date , end_date):
      data = yf.download(f"{ticker}.NS", start=start_date, end=end_date)

      # Flatten the MultiIndex columns
      data.columns = ['_'.join(col).strip() for col in data.columns.values]

      # Detect Bullish Engulfing pattern
      data['Bullish_Engulfing'] = talib.CDLENGULFING(
          data[f'Open_{ticker}.NS'].values, data[f'High_{ticker}.NS'].values, data[f'Low_{ticker}.NS'].values, data[f'Close_{ticker}.NS'].values
      )

      # Detect Morning Star pattern
      data['Morning_Star'] = talib.CDLMORNINGSTAR(
          data[f'Open_{ticker}.NS'].values, data[f'High_{ticker}.NS'].values, data[f'Low_{ticker}.NS'].values, data[f'Close_{ticker}.NS'].values
      )

      # Detect 3 white shoulder pattern
      data['3_White_Soldiers'] = talib.CDL3WHITESOLDIERS(
          data[f'Open_{ticker}.NS'].values, data[f'High_{ticker}.NS'].values, data[f'Low_{ticker}.NS'].values, data[f'Close_{ticker}.NS'].values
      )

      # Detect Evening star pattern
      data['Evening_Star'] = talib.CDLEVENINGSTAR(
          data[f'Open_{ticker}.NS'].values, data[f'High_{ticker}.NS'].values, data[f'Low_{ticker}.NS'].values, data[f'Close_{ticker}.NS'].values
      )

      # Detect 3 wblack shoulder pattern
      data['3_Black_Crows'] = talib.CDL3BLACKCROWS(
          data[f'Open_{ticker}.NS'].values, data[f'High_{ticker}.NS'].values, data[f'Low_{ticker}.NS'].values, data[f'Close_{ticker}.NS'].values
      )


      # Filter only rows where the pattern was found (non-zero)
      patterns = data[(data['Bullish_Engulfing'] != 0) | (data['Morning_Star'] != 0)]

      recent_date = patterns.tail(10)
      one_month_ago = pd.to_datetime(date.today() - relativedelta(months=1))
      recent_date = recent_date[recent_date.index >= one_month_ago]
      # print(recent_date)

      temp_dict = {}

      for index, row in recent_date.iterrows():
          key = index.strftime('%Y-%m-%d')  # Format the date
          value = [row['Bullish_Engulfing'], row['Morning_Star'], row['3_White_Soldiers'], row['Evening_Star'], row['3_Black_Crows']]
          temp_dict[key] = value

      return temp_dict

# +100 → bullish pattern detected
# −100 → bearish pattern detected
# 0 → no pattern

df['pattern_BE_MS_3WS_ES_3BC'] = df['Symbol'].apply(lambda x: get_pattern(x, start_date, end_date))
print(df.sample(5))
df['trend'].value_counts()

path = today.strftime("%d-%m-%Y") + "/" + "filtered_" + title
print(path)
df.to_csv(path, index=False)

Successfully loaded MACD_Bullish.csv


/tmp/ipython-input-4183580835.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(f"{ticker}.NS", start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-4183580835.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(f"{ticker}.NS", start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-4183580835.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(f"{ticker}.NS", start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-4183580835.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(f"{ticker}.NS", start=start_date, end=end_date)
[*********************100%***********************]  1 of 

   Sr.                               Stock Name      Symbol      Links  \
0    1                   Amd Industries Limited      AMDIND  P&F | F.A   
1    2                       Jindal Saw Limited   JINDALSAW  P&F | F.A   
2    3             Shree Rama Newsprint Limited    RAMANEWS  P&F | F.A   
3    4  Advani Hotels & Resorts (india) Limited  ADVANIHOTR  P&F | F.A   
4    5                              Blb Limited  BLBLIMITED  P&F | F.A   

    % Chg   Price        Volume  trend  
0  19.99%   51.80      6,06,710      0  
1  15.94%  179.29  12,24,65,614      1  
2   12.6%   34.49      3,00,482      1  
3   9.93%   60.36      4,16,763      1  
4   9.36%   14.60     24,69,552      0  


[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-4183580835.py:58: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(f"{ticker}.NS", start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-4183580835.py:58: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(f"{ticker}.NS", start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-4183580835.py:58: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(f"{ticker}.NS", start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-4183580835.py:58: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(f"{ticker}.NS", start=start_date, en

    Sr.                          Stock Name      Symbol      Links   % Chg  \
25   26      Intrasoft Technologies Limited        ISFT  P&F | F.A   2.28%   
41   42    Texmo Pipes And Products Limited  TEXMOPIPES  P&F | F.A  -0.06%   
49   50  Axis Technology ETF Regular Growth  AXISTECETF  P&F | F.A  -0.23%   
1     2                  Jindal Saw Limited   JINDALSAW  P&F | F.A  15.94%   
6     7                   Dic India Limited      DICIND  P&F | F.A    7.7%   

     Price        Volume  trend  \
25   93.12        46,612      0   
41   48.00        30,479      0   
49  423.99         3,751      1   
1   179.29  12,24,65,614      1   
6   502.70        12,601      0   

                             pattern_BE_MS_3WS_ES_3BC  
25  {'2025-12-19': [-100.0, 0.0, 0.0, 0.0, 0.0], '...  
41  {'2025-12-26': [-100.0, 0.0, 0.0, 0.0, 0.0], '...  
49  {'2025-12-24': [-100.0, 0.0, 0.0, 0.0, 0.0], '...  
1   {'2025-12-26': [-100.0, 0.0, 0.0, 0.0, 0.0], '...  
6   {'2025-12-22': [80.0, 0.0, 0.0, 0.0,

In [ ]:
import pandas as pd
from datetime import date

today = date.today().strftime("%d-%m-%Y")
path = today + "/" + "filtered_" + title
df = pd.read_csv(path)
print(df.head())

price = 3000

df = df[(df["Price"] < price) & (df["trend"] == 1)] # Corrected to filter for bullish trend (trend == 0)

df.to_csv(today + "/" + "filtered_price_" + title, index=False)
if not df.empty: # Add check to see if the DataFrame is empty
  print(df.sample(min(len(df), 5))) # Sample at most 5 rows, or fewer if less than 5 are available
  print(f"Total stocks found: {len(df)}")
else:
  print("No stocks found matching the criteria.")

   Sr.                               Stock Name      Symbol      Links  \
0    1                   Amd Industries Limited      AMDIND  P&F | F.A   
1    2                       Jindal Saw Limited   JINDALSAW  P&F | F.A   
2    3             Shree Rama Newsprint Limited    RAMANEWS  P&F | F.A   
3    4  Advani Hotels & Resorts (india) Limited  ADVANIHOTR  P&F | F.A   
4    5                              Blb Limited  BLBLIMITED  P&F | F.A   

    % Chg   Price        Volume  trend  \
0  19.99%   51.80      6,06,710      0   
1  15.94%  179.29  12,24,65,614      1   
2   12.6%   34.49      3,00,482      1   
3   9.93%   60.36      4,16,763      1   
4   9.36%   14.60     24,69,552      0   

                            pattern_BE_MS_3WS_ES_3BC  
0  {'2025-12-26': [np.float64(100.0), np.float64(...  
1  {'2025-12-26': [np.float64(-100.0), np.float64...  
2  {'2025-12-23': [np.float64(-100.0), np.float64...  
3  {'2025-12-24': [np.float64(-100.0), np.float64...  
4  {'2025-12-22': [np.float

In [ ]:
stocks = df["Symbol"]
filtered_stocks = stocks.to_list()
print(filtered_stocks)

['JINDALSAW', 'RAMANEWS', 'ADVANIHOTR', 'DIGJAMLMTD', 'FEDERALBNK', 'ONEPOINT', 'TECHM', 'AURUM', 'SAGILITY', 'AXISBPSETF', 'TECH', 'SHRINGARMS', 'IT', 'CANBK', 'AXISTECETF', 'SBIETFIT', 'HDFCNIFIT', 'ITETF', 'ITBEES', 'ITIETF', 'INFY', 'ITADD']


In [ ]:
#step 3 : RSI, MACD, ADX calculation
import yfinance as yf
import pandas_ta as ta


def get_technical_analysis(ticker):
        # Choose a stock symbol
        symbol = ticker + ".NS"  # Example: Apple Inc.

        # Download historical stock data
        stock_data = yf.download(symbol, period='max', interval='1d')
        if stock_data.empty:
            print(f"No data found for {symbol}.")
            return
        # Display the first few rows of the DataFrame
        # print(f"Displaying the first few rows of historical data for {symbol}:")
        # print(stock_data.head())

        # print(f"\nDisplaying the last few rows of historical data for {symbol}:")
        # print(stock_data.tail())


        # Flatten the MultiIndex columns
        # The yfinance dataframe has columns like ('Open', 'KTKBANK.NS'), ('High', 'KTKBANK.NS'), etc.
        # We want to access them as 'Open', 'High', etc. by removing the first level of the MultiIndex if it's redundant
        stock_data.columns = [col[0] if isinstance(col, tuple) else col for col in stock_data.columns]

        # Now, rename columns to standard lowercased names if pandas_ta expects them
        stock_data.columns = stock_data.columns.str.lower()

        # Calculate RSI
        stock_data.ta.rsi(append=True)

        # Calculate MACD
        stock_data.ta.macd(append=True)

        # Calculate ADX
        stock_data.ta.adx(append=True)

        # Display the last few rows with the new indicators
        # print("\nDisplaying the last few rows of stock_data with technical indicators:")
        # print(stock_data.tail())
        print(stock_data.tail(1))

        return dict(stock_data.tail(1).iloc[0])


today_folder = date.today().strftime("%d-%m-%Y")
path = today_folder + "/" + "filtered_price_" + title
# path = today_folder + "/" +"filtered_MACD_Bullish.csv"
print(path)

try:
    df = pd.read_csv(path)
    print(f"Successfully loaded {path}")
except FileNotFoundError:
    print(f"Error: The file '{path}' was not found.")
    # Handle the error, maybe exit or set df to None
    df = None # Set df to None or an empty DataFrame


if df is not None:
    # Using asyncio.run within apply - less efficient for large DFs
    df["technical"] = df['Symbol'].apply(lambda x: get_technical_analysis(x))
    print(df.sample(min(len(df), 5))) # Fixed: Sample at most 5 rows, or fewer if less than 5 are available

    today_folder = date.today().strftime("%d-%m-%Y")
    output_csv_filename = f"final_{title}"
    # output_csv_filename = f"final_{title}2"
    output_csv_path = os.path.join(today_folder, output_csv_filename)
    df.to_csv(output_csv_path, index=False)

/tmp/ipython-input-1706843857.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(symbol, period='max', interval='1d')


19-01-2026/filtered_price_MACD_Bullish.csv
Successfully loaded 19-01-2026/filtered_price_MACD_Bullish.csv


[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-1706843857.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(symbol, period='max', interval='1d')


                 close        high    low   open     volume    RSI_14  \
Date                                                                    
2026-01-19  179.289993  183.399994  155.0  155.0  122450409  63.85401   

            MACD_12_26_9  MACDh_12_26_9  MACDs_12_26_9     ADX_14  ADXR_14_2  \
Date                                                                           
2026-01-19     -0.279438        0.23485      -0.514288  21.886188  21.421737   

               DMP_14     DMN_14  
Date                              
2026-01-19  31.379412  19.200122  


[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-1706843857.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(symbol, period='max', interval='1d')


...

            close       high    low       open  volume     RSI_14  \
Date                                                                
2026-01-19  42.82  43.049999  42.25  43.049999  649985  58.903455   

            MACD_12_26_9  MACDh_12_26_9  MACDs_12_26_9     ADX_14  ADXR_14_2  \
Date                                                                           
2026-01-19      0.208483       0.030152        0.17833  13.893415  13.354883   

              DMP_14    DMN_14  
Date                            
2026-01-19  2.402149  1.580357  


[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-1706843857.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(symbol, period='max', interval='1d')


                  close    high     low    open   volume     RSI_14  \
Date                                                                  
2026-01-19  1681.199951  1685.0  1666.5  1685.0  5104784  63.610478   

            MACD_12_26_9  MACDh_12_26_9  MACDs_12_26_9     ADX_14  ADXR_14_2  \
Date                                                                           
2026-01-19     11.707872       2.308575       9.399297  23.016715  22.244505   

                DMP_14     DMN_14  
Date                               
2026-01-19  141.928891  72.006892  


[*********************100%***********************]  1 of 1 completed

                close       high        low       open  volume     RSI_14  \
Date                                                                        
2026-01-19  40.810001  41.849998  40.610001  41.060001   11145  58.491226   

            MACD_12_26_9  MACDh_12_26_9  MACDs_12_26_9     ADX_14  ADXR_14_2  \
Date                                                                           
2026-01-19      0.211242       0.021371        0.18987  12.642983  11.522284   

             DMP_14    DMN_14  
Date                           
2026-01-19  3.55851  1.854006  
    Sr.                                         Stock Name      Symbol  \
0     2                                 Jindal Saw Limited   JINDALSAW   
21   58                                   DSP Nifty IT ETF       ITADD   
14   50                 Axis Technology ETF Regular Growth  AXISTECETF   
9    38  Axis AAA Bond Plus SDL ETF-2026 Matur. Reg. Gr...  AXISBPSETF   
16   52                                  HDFC Nifty IT ETF   

In [ ]:
df["technical"][0]

{'close': np.float64(179.2899932861328),
 'high': np.float64(183.39999389648438),
 'low': np.float64(155.0),
 'open': np.float64(155.0),
 'volume': np.float64(122450409.0),
 'RSI_14': np.float64(63.8540103499626),
 'MACD_12_26_9': np.float64(-0.27943849985643965),
 'MACDh_12_26_9': np.float64(0.234849771147001),
 'MACDs_12_26_9': np.float64(-0.5142882710034407),
 'ADX_14': np.float64(21.88618776365746),
 'ADXR_14_2': np.float64(21.421737127600462),
 'DMP_14': np.float64(31.37941199410924),
 'DMN_14': np.float64(19.20012159322199)}

In [ ]:
#step4 ; image
import asyncio
import os
import pandas as pd
from datetime import date
from playwright.async_api import async_playwright

# Configuration
SCAN_TITLE = "MACD_Bullish"  # Corrected to match the generated CSV file
VIEWPORT = {"width": 1280, "height": 800}
WAIT_TIME = 3000 # Time in ms to wait for chart to render

async def capture_chart(ticker, folder_path):
    """
    Captures a screenshot for a specific ticker.
    """
    url = f"https://chartink.com/stocks-new?from_scan=1&scan_link=scanlink:5aaed7b79143ccbfccbfdd74bc86f3d3&timeframe=daily&symbol={ticker}"
    output_filename = f"{ticker}.png"
    full_path = os.path.join(folder_path, output_filename)

    print(f"\n⏳ Processing: {ticker}")

    async with async_playwright() as p:
        try:
            # Launch Browser
            browser = await p.chromium.launch(headless=True)
            page = await browser.new_page(viewport=VIEWPORT)

            # Navigate
            print(f"   → Opening URL for {ticker}...")
            await page.goto(url, wait_until="networkidle")

            # Wait for Chart Container
            try:
                # Waiting for common chart selectors
                await page.wait_for_selector("div.chart-container, iframe, #tv_chart_container", timeout=10000)
                print("   → Chart element detected, rendering...")
                await page.wait_for_timeout(WAIT_TIME)
            except Exception as e:
                print(f"   ⚠️ Warning: Chart container timed out for {ticker}. Taking screenshot anyway.")

            # Take Screenshot
            await page.screenshot(path=full_path, full_page=True)
            print(f"   ✅ Screenshot saved: {full_path}")

            await browser.close()
            return True

        except Exception as e:
            print(f"   ❌ Failed to capture {ticker}: {e}")
            return False

async def main_process():
    # 1. Setup Paths and Dates
    today_str = date.today().strftime("%d-%m-%Y")

    # Assuming the CSV is in a folder named after today's date
    base_dir = today_str
    screenshot_dir = os.path.join(base_dir, "ScreenShot")

    # Ensure Screenshot Directory Exists
    if not os.path.exists(screenshot_dir):
        os.makedirs(screenshot_dir)
        print(f"📁 Created directory: {screenshot_dir}")

    # 2. Load the CSV
    # Note: Ensure 'final_SCAN_TITLE.csv' exists in the today_folder
    csv_filename = f"final_{SCAN_TITLE}.csv"
    csv_path = os.path.join(base_dir, csv_filename)

    if not os.path.exists(csv_path):
        print(f"❌ Error: CSV file not found at {csv_path}")
        return

    print(f"📊 Loading data from: {csv_path}")
    df = pd.read_csv(csv_path)

    # Verify column existence
    if 'Symbol' not in df.columns:
        print("❌ Error: CSV must contain a 'Symbol' column.")
        return

    print(f"🔹 Found {len(df)} stocks to process.")
    print("-" * 40)

    # 3. Step-by-Step Execution Loop
    await capture_chart('nifty',screenshot_dir)
    success_count = 0
    for index, row in df.iterrows():
        ticker = row['Symbol']

        # Await the capture (This ensures step-by-step execution)
        result = await capture_chart(ticker, screenshot_dir)

        if result:
            success_count += 1

        # Optional: Small delay between requests to be polite to the server
        await asyncio.sleep(1)

    print("-" * 40)
    print(f"🎉 Process Complete. {success_count}/{len(df)} screenshots captured successfully.")
    print(f"📂 Files saved in: {screenshot_dir}")

# Call the async main_process function directly using await
await main_process()

📁 Created directory: 19-01-2026/ScreenShot
📊 Loading data from: 19-01-2026/final_MACD_Bullish.csv
🔹 Found 22 stocks to process.
----------------------------------------

⏳ Processing: nifty
   → Opening URL for nifty...
   ⚠️ Warning: Chart container timed out for nifty. Taking screenshot anyway.
   ✅ Screenshot saved: 19-01-2026/ScreenShot/nifty.png

⏳ Processing: JINDALSAW
   → Opening URL for JINDALSAW...
   ⚠️ Warning: Chart container timed out for JINDALSAW. Taking screenshot anyway.
   ✅ Screenshot saved: 19-01-2026/ScreenShot/JINDALSAW.png

⏳ Processing: RAMANEWS
   → Opening URL for RAMANEWS...
   ⚠️ Warning: Chart container timed out for RAMANEWS. Taking screenshot anyway.
   ✅ Screenshot saved: 19-01-2026/ScreenShot/RAMANEWS.png

⏳ Processing: ADVANIHOTR
   → Opening URL for ADVANIHOTR...
   ⚠️ Warning: Chart container timed out for ADVANIHOTR. Taking screenshot anyway.
   ✅ Screenshot saved: 19-01-2026/ScreenShot/ADVANIHOTR.png

⏳ Processing: DIGJAMLMTD
   → Opening URL for 



#  Fundamental Analysis



In [ ]:
import requests
from bs4 import BeautifulSoup
from tempfile import tempdir
import yfinance as yf

# input
# ticker = "SIGMA"

def WebScaping(ticker, ratios, charts, holdings, analysis):
          #utility functions =============================================================

          def remove_extra_space_and_characters(input_string):
            characters_to_remove = [' ','\n', '\t', '\r', '\xa0','Cr.','₹','%'] # Removed 'Promoters\xa0+'
            output = input_string.strip()
            for char in characters_to_remove:
              output = output.replace(char, '')
            # Handle 'NA' or empty strings by returning 0.0
            if output == 'NA' or output == '':
                return 0.0
            try:
                return float(output)
            except ValueError:
                print(f"Could not convert to float: {output}")
                return 0.0


          def mainContentRatio(id):
            main_body = body[0].find('div',id=id)
            kv = main_body.text.strip().replace('\n\n','').split('\n')
            key = kv[0].strip()
            value = kv[1]
            ratios[key] = remove_extra_space_and_characters(value)

          def chart_ratio(yearly_values, name):
            for year in yearly_values:
              y = year.text.strip().split('Year')
              key = name + " " + y[0] + "Year"
              value = y[1].replace('%','')
              charts[key] = float(value)


          #TICKER body extract ==============================================================
          url = f'https://ticker.finology.in/company/{ticker}'

          headers = {
              "User-Agent": (
                  "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/122.0.0.0 Safari/537.36"
              )
          }

          # Add try-except block to handle HTTPError
          try:
              # Get the page
              response = requests.get(url, headers=headers)
              response.raise_for_status()

              # Wrap in BeautifulSoup
              soup = BeautifulSoup(response.text, "html.parser")
              body = soup.select('body')

              #Screener extract =================================================================
              url2 = f'https://www.screener.in/company/{ticker}/consolidated/'
              response2 = requests.get(url2, headers=headers)
              response2.raise_for_status()
              soup2 = BeautifulSoup(response2.text, "html.parser")
              body2 = soup2.select('body')


              # Define the ticker symbol
              ticker_NS = ticker + ".NS"  # ".NS" is for NSE India
              stock = yf.Ticker(ticker_NS)

              # Get the industry information
              industry = stock.info.get('industry', 'Industry not found')
              long_description = stock.info.get('longBusinessSummary','Description Not found')
              ratios["Industry"] = industry
              ratios["Description"] = long_description

              #extract sector
              sector = body[0].find_all(id='mainContent_compinfoId')[0].select('a')[0].text.strip()
              ratios["Sector"] = sector

              #Price extract
              price = float(body[0].find(id="mainContent_clsprice").find_all('span')[-1].text.strip())
              ratios["Price"] = price

              main_ratio_section = body[0].find(id="companyessentials")

              inner_divs_1 = main_ratio_section.find_all('div', recursive=False)
              inner_divs_2 = inner_divs_1[1].find_all('div', recursive=False)

              quick_ratio = stock.info.get('quickRatio')
              ratios["QuickRatio"] = quick_ratio

              trailingPegRatio = stock.info.get('trailingPegRatio')
              ratios["PEG"] = trailingPegRatio

              #main ratios ======================================================================
              for inner_div in inner_divs_2:
                small_tag = inner_div.find('small')
                if small_tag is not None:
                  ratio = small_tag.text.strip()
                  p_tag = inner_div.find('p')
                  if p_tag is not None:
                    ratios[ratio] = remove_extra_space_and_characters(p_tag.text.strip())

              #More raios =======================================================================
              mainContentRatio('mainContent_divCFOPAT')
              mainContentRatio('mainContent_divDebtEquity')
              mainContentRatio('mainContent_divICR')


              #charts ===========================================================================
              charts_section = body[0].find_all(class_='card cardscreen cardsmall')

              Sales_growth_chart = charts_section[0]
              chart = Sales_growth_chart.find('canvas')

              chart_ratio(charts_section[0].find_all('div',class_='ratiosingle'),'SalesGrowth')
              chart_ratio(charts_section[1].find_all('div',class_='ratiosingle'),'ROE')
              chart_ratio(charts_section[2].find_all('div',class_='ratiosingle'),'ROCE')
              chart_ratio(charts_section[3].find_all('div',class_='ratiosingle'),'ProfitGrowth')


              #share-holding-pattern ============================================================
              Shareholding_Pattern = body[0].find(id="mainContent_DivShp").find('tbody')

              rows = Shareholding_Pattern.find_all('tr')
              for row in rows:
                cells = row.find_all('td')
                key = cells[0].text.strip()
                temp_dic = {}
                sub_key = "Promoter"
                # Modified to handle 'Promoters\xa0+' specifically before converting to float
                value = cells[1].text.strip().replace('Promoters\xa0+', '').replace('%', '').replace(',', '')
                temp_dic[sub_key] = float(value) if value else 0.0 # Convert to float only if value is not empty
                sub_key = "Pledge"
                value = cells[2].text.strip().replace('%', '').replace(',', '')
                temp_dic[sub_key] = float(value) if value else 0.0 # Convert to float only if value is not empty
                holdings[key] = temp_dic


              #More-share-holding-pattern =======================================================
              section = body2[0].find(id="shareholding")
              table = section.find('tbody')
              header = section.find('thead')
              cols = header.find_all('th')

              keys = []
              for col in cols:
                keys.append(col.text.strip())

              keys = keys[-5:]

              rows = table.find_all('tr')

              for data in rows:
                recent_data = data.find_all('td')
                first = recent_data[0].text.strip().split('\n')[0].replace('\xa0+','')
                last_five = recent_data[-5:]

                # epochs = 5 # Removed fixed epochs
                temp_dic = {}
                # Iterate only up to the length of last_five
                for epoch in range(len(last_five)):
                  # Modified to handle potential empty strings after replacement and the specific string
                  value_str = last_five[epoch].text.strip().replace('%','').replace(',','').replace('Promoters\xa0+', '')
                  # Check if keys[epoch] is a valid non-empty string before using it as a key
                  if keys and epoch < len(keys) and keys[epoch]:
                      holdings[keys[epoch]][first] = float(value_str) if value_str else 0.0
                  else:
                      # Added a print statement to help diagnose which key is causing the issue
                      print(f"Warning: Skipping data for {first} due to invalid key '{keys[epoch] if epoch < len(keys) else 'Index out of bounds'}' at epoch {epoch}")


              #strengt-and-limitation ===========================================================
              strength_and_limitaion_section = body[0].find_all('div',class_='card cardscreenFixed overflow-hidden')
              strength_section = strength_and_limitaion_section[0]
              limitation_section = strength_and_limitaion_section[1]

              strength = []
              limitation = []

              for li in strength_section.find_all('ul')[0].find_all('li'):
                strength.append(li.text.strip())

              for li in limitation_section.find_all('ul')[0].find_all('li'):
                limitation.append(li.text.strip())

              strength_and_limitaion_section2 = body2[0].find('section',id='analysis')
              strength_and_limitaion_section2

              pros = strength_and_limitaion_section2.find('div',class_='pros').find_all('li')
              cons = strength_and_limitaion_section2.find('div',class_='cons').find_all('li')

              for pro in pros:
                strength.append(pro.text.strip())

              for con in cons:
                limitation.append(con.text.strip())

              analysis["Strength"] = strength
              analysis["Limitation"] = limitation


              #news =============================================================================
              # News_section = body[0].find_all("div",class_="card cardscreen")[-1]
              # News_pointers = News_section.find_all("p")
              # News_section2 = body[0].find_all("div",class_="card cardscreen")[-2]
              # News_pointers2 = News_section.find_all("li")
              # News_pointers = News_pointers + News_pointers2
              # pointers = []
              # for pointer in News_pointers:
              #   pointers.append(pointer.text.strip().replace('\xa0',''))

              # analysis["NEWS"] = pointers

          except requests.exceptions.HTTPError as e:
              print(f"HTTP Error occurred for ticker {ticker}: {e}")
              # Return empty dictionaries if HTTPError occurs
              return {}, {}, {}, {}
          except Exception as e:
              print(f"An unexpected error occurred for ticker {ticker}: {e}")
              # Return empty dictionaries for any other unexpected errors
              return {}, {}, {}, {}


          print(ratios)
          print(charts)
          print(holdings)
          print(analysis)

          return ratios,charts, holdings, analysis

In [ ]:
def calculate_fundamental_score(ratios, charts, holdings, analysis,
                                  govt_bond_yield=6.53,
                                  historical_pe_avg=None,
                                  sector_pe_avg=None):
    """
    Calculate fundamental stock score based on industry-standard weighted criteria.

    CORRECTIONS MADE:
    1. Fixed profit/sales growth to use absolute values, not deltas
    2. Corrected holding score formula and pledge penalty
    3. Rebalanced weights based on industry standards
    4. Added earnings yield to scoring
    5. Made PEG and Quick Ratio handling consistent
    6. Improved ROE thresholds
    7. Enhanced P/E scoring with growth consideration
    8. **Added handling for empty holdings data.**
    """

    asset_heavy_industries = [
        "Aerospace & Defense", "Aluminum", "Auto Manufacturers", "Building Materials",
        "Chemicals", "Specialty Chemicals", "Copper", "Steel", "Oil & Gas E&P",
        "Oil & Gas Equipment & Services", "Oil & Gas Integrated", "Oil & Gas Midstream",
        "Oil & Gas Refining & Marketing", "Thermal Coal", "Solar", "Utilities - Diversified",
        "Utilities - Regulated Electric", "Utilities - Regulated Gas", "Utilities - Regulated Water",
        "Farm & Heavy Construction Machinery", "Engineering & Construction", "Metal Fabrication",
        "Specialty Industrial Machinery", "Marine Shipping", "Railroads", "Trucking", "Airlines",
        "Integrated Freight & Logistics", "Real Estate - Development", "REIT - Diversified",
        "REIT - Healthcare Facilities", "REIT - Hotel & Motel", "REIT - Industrial",
        "REIT - Office", "REIT - Residential", "REIT - Retail", "REIT - Specialty",
        "Banks - Diversified", "Banks - Regional", "Power"
    ]

    import numpy as np

    # CORRECTED: Industry-standard weight configuration (total = 100)
    weights = {
        'Profit Growth': 15,        # Increased from 10
        'Sales Growth': 8,          # Reduced from 10
        'Earnings Yield': 5,        # NEW - was not scored before
        'P/E Ratio': 10,            # Increased from 8
        'PEG Ratio': 8,             # Same
        'Debt-to-Equity': 8,        # Increased from 7
        'ROE & Dividend': 10,       # Increased from 10
        'ROCE': 10,                  # Increased from 7
        'P/B Ratio & Book Value': 6, # Same
        'Promoter/DII/FII Holding': 8, # Same
        'Interest Coverage': 5,     # Reduced from 7
        'Quick Ratio': 5,           # Same
        'CFO/PAT Ratio': 10        # Reduced from 12
    }

    scores = {}
    details = {}

    # Extract key values
    price = ratios.get('Price', 0)
    eps = ratios.get('EPS (TTM)', 0)
    current_pe = ratios.get('P/E', 0)
    peg_ratio = ratios.get('PEG', None)
    debt_to_equity = ratios.get('Debt/Equity', 0)
    roe = ratios.get('ROE', 0)
    roce = ratios.get('ROCE', 0)
    div_yield = ratios.get('Div. Yield', 0)
    pb_ratio = ratios.get('P/B', 0)
    book_value = ratios.get('Book Value (TTM)', 0)
    interest_coverage = ratios.get('Interest Cover Ratio', 0)
    quick_ratio = ratios.get('QuickRatio', None)
    cfo_pat_ratio = ratios.get('CFO/PAT (5 Yr. Avg.)', 0)
    industry = ratios.get('Industry', 'Unknown')

    # Get holdings - Added handling for empty holdings
    promoter_holding = 0
    fii_holding = 0
    dii_holding = 0
    pledge = 0.0

    if holdings:
        latest_quarter = list(holdings.keys())[0]
        latest_holdings = holdings.get(latest_quarter, {})
        promoter_holding = latest_holdings.get('Promoters', 0)
        fii_holding = latest_holdings.get('FIIs', 0)
        dii_holding = latest_holdings.get('DIIs', 0)
        pledge = float(latest_holdings.get('Pledge', 0))


    # Get growth metrics
    curr_profit_growth = ratios.get('Profit Growth', 0)
    curr_sales_growth = ratios.get('Sales Growth', 0)

    # ===========================================
    # 1. EARNINGS YIELD (NEW - Weight: 5)
    # ===========================================
    earnings_yield = (eps / price * 100) if price > 0 else 0
    earnings_vs_bond = earnings_yield > govt_bond_yield

    if earnings_yield > govt_bond_yield * 1.5:
        ey_score = 100
    elif earnings_yield > govt_bond_yield * 1.2:
        ey_score = 80
    elif earnings_yield > govt_bond_yield:
        ey_score = 60
    elif earnings_yield > govt_bond_yield * 0.8:
        ey_score = 40
    else:
        ey_score = 20

    scores['Earnings Yield'] = (ey_score / 100) * weights['Earnings Yield']
    details['Earnings Yield'] = {
        'value': earnings_yield,
        'bond_yield': govt_bond_yield,
        'undervalued': earnings_vs_bond,
        'score_pct': ey_score
    }

    # ===========================================
    # 2. PROFIT GROWTH - CORRECTED (Weight: 15)
    # ===========================================
    # Use absolute current growth rate, not delta
    if curr_profit_growth >= 25:
        profit_growth_score = 100
    elif curr_profit_growth >= 20:
        profit_growth_score = 90
    elif curr_profit_growth >= 15:
        profit_growth_score = 75
    elif curr_profit_growth >= 10:
        profit_growth_score = 55
    elif curr_profit_growth >= 5:
        profit_growth_score = 35
    elif curr_profit_growth >= 0:
        profit_growth_score = 20
    else:
        profit_growth_score = 0  # Negative growth

    scores['Profit Growth'] = (profit_growth_score / 100) * weights['Profit Growth']
    details['Profit Growth'] = {'value': curr_profit_growth, 'score_pct': profit_growth_score}

    # ===========================================
    # 3. SALES GROWTH - CORRECTED (Weight: 8)
    # ===========================================
    if curr_sales_growth >= 25:
        sales_growth_score = 100
    elif curr_sales_growth >= 20:
        sales_growth_score = 90
    elif curr_sales_growth >= 15:
        sales_growth_score = 75
    elif curr_sales_growth >= 10:
        sales_growth_score = 55
    elif curr_sales_growth >= 5:
        sales_growth_score = 35
    elif curr_sales_growth >= 0:
        sales_growth_score = 20
    else:
        sales_growth_score = 0

    scores['Sales Growth'] = (sales_growth_score / 100) * weights['Sales Growth']
    details['Sales Growth'] = {'value': curr_sales_growth, 'score_pct': sales_growth_score}

    # ===========================================
    # 4. P/E RATIO - IMPROVED (Weight: 10)
    # ===========================================
    if historical_pe_avg and sector_pe_avg:
        pe_discount = ((historical_pe_avg - current_pe) / historical_pe_avg) * 100

        if current_pe < sector_pe_avg * 0.8 and current_pe < historical_pe_avg * 0.8:
            pe_score = 100
        elif current_pe < min(sector_pe_avg, historical_pe_avg):
            pe_score = 85
        elif current_pe < historical_pe_avg * 1.1:
            pe_score = 70
        elif current_pe < historical_pe_avg * 1.2:
            pe_score = 55
        else:
            pe_score = 30
    else:
        # Enhanced default scoring considering growth
        if current_pe < 12:
            pe_score = 100
        elif current_pe < 18:
            pe_score = 85
        elif current_pe < 25:
            pe_score = 70
        elif current_pe < 30:
            pe_score = 50
        elif current_pe < 40:
            pe_score = 30
        else:
            pe_score = 10

    scores['P/E Ratio'] = (pe_score / 100) * weights['P/E Ratio']
    details['P/E Ratio'] = {
        'current': current_pe,
        'historical_avg': historical_pe_avg,
        'sector_avg': sector_pe_avg,
        'score_pct': pe_score
    }

    # ===========================================
    # 5. PEG RATIO - FIXED (Weight: 8)
    # ===========================================
    if peg_ratio is not None and peg_ratio > 0:
        if peg_ratio < 0.5:
            peg_score = 100
        elif peg_ratio < 1:
            peg_score = 90
        elif peg_ratio < 1.5:
            peg_score = 70
        elif peg_ratio < 2:
            peg_score = 50
        elif peg_ratio < 3:
            peg_score = 30
        else:
            peg_score = 10
    else:
        # Default score if PEG not available
        peg_score = 50

    scores['PEG Ratio'] = (peg_score / 100) * weights['PEG Ratio']
    details['PEG Ratio'] = {'value': peg_ratio, 'score_pct': peg_score}

    # ===========================================
    # 6. DEBT-TO-EQUITY (Weight: 8)
    # ===========================================
    if debt_to_equity == 0:
        de_score = 100
    elif debt_to_equity < 0.25:
        de_score = 95
    elif debt_to_equity < 0.5:
        de_score = 85
    elif debt_to_equity < 1:
        de_score = 65
    elif debt_to_equity < 1.5:
        de_score = 45
    elif debt_to_equity < 2:
        de_score = 30
    else:
        de_score = 10

    scores['Debt-to-Equity'] = (de_score / 100) * weights['Debt-to-Equity']
    details['Debt-to-Equity'] = {'value': debt_to_equity, 'score_pct': de_score}

    # ===========================================
    # 7. ROE & DIVIDEND - IMPROVED (Weight: 12)
    # ===========================================
    if debt_to_equity < 0.5:
        # Enhanced ROE thresholds
        if roe > 20 and div_yield < 3:
            roe_div_score = 100
        elif roe > 18 and div_yield < 3:
            roe_div_score = 90
        elif roe > 15 and div_yield < 3:
            roe_div_score = 80
        elif roe > 15 and div_yield < 5:
            roe_div_score = 65
        elif roe > 12 and div_yield < 5:
            roe_div_score = 50
        elif roe > 10:
            roe_div_score = 35
        else:
            roe_div_score = 10

        scores['ROE & Dividend'] = (roe_div_score / 100) * weights['ROE & Dividend']
        details['ROE & Dividend'] = {
            'roe': roe,
            'div_yield': div_yield,
            'score_pct': roe_div_score
        }
    else:
        # Use ROCE for high debt companies
        if roce > 20:
            roce_score = 100
        elif roce > 18:
            roce_score = 90
        elif roce > 15:
            roce_score = 75
        elif roce > 12:
            roce_score = 55
        elif roce > 10:
            roce_score = 35
        else:
            roce_score = 10

        scores['ROCE'] = (roce_score / 100) * weights['ROCE']
        details['ROCE'] = {'value': roce, 'score_pct': roce_score}

    # ===========================================
    # 8. P/B RATIO & BOOK VALUE (Weight: 6)
    # ===========================================
    if industry in asset_heavy_industries:
        if pb_ratio < 0.8:
            pb_score = 100
        elif pb_ratio < 1:
            pb_score = 95
        elif pb_ratio < 1.5:
            pb_score = 85
        elif pb_ratio < 2:
            pb_score = 70
        elif pb_ratio < 3:
            pb_score = 50
        elif pb_ratio < 5:
            pb_score = 30
        else:
            pb_score = 10

        scores['P/B Ratio & Book Value'] = (pb_score / 100) * weights['P/B Ratio & Book Value']
        details['P/B Ratio & Book Value'] = {
            'pb_ratio': pb_ratio,
            'book_value': book_value,
            'price': price,
            'industry': industry,
            'asset_heavy': True,
            'score_pct': pb_score
        }
    else:
        # For asset-light companies, P/B is less relevant
        pb_score = 50  # Neutral score
        scores['P/B Ratio & Book Value'] = (pb_score / 100) * weights['P/B Ratio & Book Value']
        details['P/B Ratio & Book Value'] = {
            'pb_ratio': pb_ratio,
            'book_value': book_value,
            'price': price,
            'industry': industry,
            'asset_heavy': False,
            'score_pct': pb_score
        }

    # ===========================================
    # 9. HOLDING PATTERN - CORRECTED (Weight: 8)
    # ===========================================
    # Added check for empty holdings
    if holdings:
        institutional_holding = fii_holding + dii_holding

        # Promoter holding score (40% weight)
        if 50 <= promoter_holding <= 75:
            promoter_score = 100
        elif 40 <= promoter_holding < 50 or 75 < promoter_holding <= 80:
            promoter_score = 85
        elif 30 <= promoter_holding < 40 or 80 < promoter_holding <= 85:
            promoter_score = 65
        elif promoter_holding < 30:
            promoter_score = 40
        else:  # > 85%
            promoter_score = 50  # Too high can limit liquidity

        # Institutional holding score (60% weight)
        if institutional_holding >= 30:
            institutional_score = 100
        elif institutional_holding >= 25:
            institutional_score = 85
        elif institutional_holding >= 20:
            institutional_score = 70
        elif institutional_holding >= 15:
            institutional_score = 50
        else:
            institutional_score = 30

        # Combined holding score
        holding_score = (0.4 * promoter_score + 0.6 * institutional_score)

        # CORRECTED: Pledge penalty
        if pledge > 0:
            if pledge < 5:
                pledge_multiplier = 0.95
            elif pledge < 10:
                pledge_multiplier = 0.85
            elif pledge < 20:
                pledge_multiplier = 0.70
            elif pledge < 30:
                pledge_multiplier = 0.50
            else:
                pledge_multiplier = 0.25
            holding_score = holding_score * pledge_multiplier
    else:
        # Default score if holdings is empty
        holding_score = 0

    scores['Promoter/DII/FII Holding'] = (holding_score / 100) * weights['Promoter/DII/FII Holding']
    details['Promoter/DII/FII Holding'] = {
        'promoter': promoter_holding,
        'fii': fii_holding,
        'dii': dii_holding,
        'institutional': fii_holding + dii_holding if holdings else 0, # Set to 0 if holdings empty
        'pledge': pledge,
        'score_pct': holding_score
    }

    # ===========================================
    # 10. INTEREST COVERAGE (Weight: 5)
    # ===========================================
    if interest_coverage > 15:
        ic_score = 100
    elif interest_coverage > 10:
        ic_score = 90
    elif interest_coverage > 7:
        ic_score = 75
    elif interest_coverage > 5:
        ic_score = 60
    elif interest_coverage > 3:
        ic_score = 40
    elif interest_coverage > 2:
        ic_score = 25
    else:
        ic_score = 10

    scores['Interest Coverage'] = (ic_score / 100) * weights['Interest Coverage']
    details['Interest Coverage'] = {'value': interest_coverage, 'score_pct': ic_score}

    # ===========================================
    # 11. QUICK RATIO - FIXED (Weight: 5)
    # ===========================================
    if quick_ratio is not None:
        if quick_ratio > 2:
            qr_score = 100
        elif quick_ratio > 1.5:
            qr_score = 90
        elif quick_ratio > 1.2:
            qr_score = 75
        elif quick_ratio > 1:
            qr_score = 60
        elif quick_ratio > 0.8:
            qr_score = 40
        elif quick_ratio > 0.5:
            qr_score = 25
        else:
            qr_score = 10
    else:
        qr_score = 50  # Default if not available

    scores['Quick Ratio'] = (qr_score / 100) * weights['Quick Ratio']
    details['Quick Ratio'] = {'value': quick_ratio, 'score_pct': qr_score}

    # ===========================================
    # 12. CFO/PAT RATIO (Weight: 10)
    # ===========================================
    if 0.95 <= cfo_pat_ratio <= 1.05:
        cfo_pat_score = 100
    elif 0.9 <= cfo_pat_ratio < 0.95 or 1.05 < cfo_pat_ratio <= 1.1:
        cfo_pat_score = 90
    elif 0.85 <= cfo_pat_ratio < 0.9 or 1.1 < cfo_pat_ratio <= 1.15:
        cfo_pat_score = 75
    elif 0.8 <= cfo_pat_ratio < 0.85 or 1.15 < cfo_pat_ratio <= 1.2:
        cfo_pat_score = 60
    elif 0.7 <= cfo_pat_ratio < 0.8 or 1.2 < cfo_pat_ratio <= 1.3:
        cfo_pat_score = 40
    else:
        cfo_pat_score = 20

    scores['CFO/PAT Ratio'] = (cfo_pat_score / 100) * weights['CFO/PAT Ratio']
    details['CFO/PAT Ratio'] = {'value': cfo_pat_ratio, 'score_pct': cfo_pat_score}

    # ===========================================
    # CALCULATE TOTAL SCORE
    # ===========================================
    total_score = sum(scores.values())
    max_possible_score = sum(weights.values())
    score_percentage = (total_score / max_possible_score) * 100

    # Rating based on corrected scale
    if score_percentage >= 85:
        rating = "EXCELLENT - Strong Buy"
        risk_level = "Low"
    elif score_percentage >= 75:
        rating = "VERY GOOD - Buy"
        risk_level = "Low to Moderate"
    elif score_percentage >= 65:
        rating = "GOOD - Consider Buy"
        risk_level = "Moderate"
    elif score_percentage >= 55:
        rating = "ABOVE AVERAGE - Hold"
        risk_level = "Moderate"
    elif score_percentage >= 45:
        rating = "AVERAGE - Caution"
        risk_level = "Moderate to High"
    elif score_percentage >= 35:
        rating = "BELOW AVERAGE - Avoid"
        risk_level = "High"
    else:
        rating = "POOR - Strong Avoid"
        risk_level = "Very High"

    # Identify strengths and weaknesses
    strengths = []
    weaknesses = []

    for criterion, weight_value in weights.items():
        if criterion in scores:
            score_value = scores[criterion]
            efficiency = (score_value / weight_value) * 100

            if efficiency >= 80:
                strengths.append({
                    'criterion': criterion,
                    'score': round(score_value, 2),
                    'weight': weight_value,
                    'efficiency': round(efficiency, 1)
                })
            elif efficiency < 50:
                weaknesses.append({
                    'criterion': criterion,
                    'score': round(score_value, 2),
                    'weight': weight_value,
                    'efficiency': round(efficiency, 1)
                })

    return {
        'total_score': round(total_score, 2),
        'max_score': max_possible_score,
        'score_percentage': round(score_percentage, 1),
        'rating': rating,
        'risk_level': risk_level,
        'scores': {k: round(v, 2) for k, v in scores.items()},
        'details': details,
        'weights': weights,
        'strengths': sorted(strengths, key=lambda x: x['efficiency'], reverse=True),
        'weaknesses': sorted(weaknesses, key=lambda x: x['efficiency']),
        # 'news' : analysis['NEWS']
    }


# Test the function with TCS data
# result = calculate_fundamental_score(
#     ratios=ratios,
#     charts=charts,
#     holdings=holdings,
#     analysis=analysis,
#     govt_bond_yield=6.53,
#     historical_pe_avg=26.42,
#     sector_pe_avg=22.0
# )

# print("\n" + "="*80)
# print("FUNCTION TEST - COMPLETE RESULTS")
# print("="*80)
# print(f"\nTotal Score: {result['total_score']:.2f}/{result['max_score']}")
# print(f"Percentage: {result['score_percentage']:.1f}%")
# print(f"Rating: {result['rating']}")
# print(f"Risk Level: {result['risk_level']}")
# print(f"\nNumber of Strengths: {len(result['strengths'])}")
# print(f"Number of Weaknesses: {len(result['weaknesses'])}")

# print("\n✓ Function created successfully and tested!")
# print("  Use: result = calculate_fundamental_score(ratios, charts, holdings, analysis)")

In [ ]:
from datetime import datetime
import pandas as pd

best_stock_ticker = []

def fundamental_Analysis(ticker):
      print(ticker)
      #dict =============================================================================
      ratios = {}
      charts = {}
      holdings = {}
      analysis = {}

      ratios,charts, holdings, analysis = WebScaping(ticker, ratios, charts, holdings, analysis)

      print(f"\nTicker: {ticker} fundamental done")

      result = calculate_fundamental_score(
          ratios=ratios,
          charts=charts,
          holdings=holdings,
          analysis=analysis,
          govt_bond_yield=6.53,
          historical_pe_avg=26.42,
          sector_pe_avg=22.0
      )
      print("\n" + "="*80)
      print("FUNCTION TEST - COMPLETE RESULTS")
      print("="*80)
      print(f"\nTotal Score: {result['total_score']:.2f}/{result['max_score']}")
      print(f"Percentage: {result['score_percentage']:.1f}%")
      if result['score_percentage'] > 50:
        best_stock_ticker.append(ticker)
      print(f"Rating: {result['rating']}")
      print(f"Risk Level: {result['risk_level']}")
      print(f"\nNumber of Strengths: {len(result['strengths'])}")
      print(f"Number of Weaknesses: {len(result['weaknesses'])}")

      print("\n✓ Function created successfully and tested!")
      print("  Use: result = calculate_fundamental_score(ratios, charts, holdings, analysis)")
      return result


file_name = "final_MACD_Bullish.csv"
path = datetime.today().strftime("%d-%m-%Y") + "/" + file_name

df = pd.read_csv(path)

df["fundametal_resullt"]=df["Symbol"].apply(fundamental_Analysis)

df.to_csv(path)

print(best_stock_ticker)

JINDALSAW
{'Industry': 'Steel', 'Description': 'Jindal Saw Limited engages in the manufacture and supply of iron and steel pipes, and pellets in India and internationally. The company offers SAW pipes, such as line pipes, double jointing, anti-corrosion and other relevant coatings, connector casings, hot pulled induction bends, monel sheating, and anodes used for energy transportation in the oil and gas sector, as well as water and slurry transportation; ductile iron pipes and fittings for water and waste-water transportation sectors; carbon, alloy, and stainless steel seamless and welded pipes and tubes for use in petroleum, exploration, sugar, steel, bearing, automotive general engineering, power, and process industries; and operates iron ore mine and pellet plant. It also provides precision stainless steel strips and soft magnetic nickel alloys for use in the production of textile machinery, clocks, watches, and electrical equipment; anti corrosion and protective coating with fittin

In [ ]:
# # Filter df to include only stocks in best_stock_ticker
# df = df[df['Symbol'].isin(best_stock_ticker)]

# # Display the filtered DataFrame (optional)
# print("Filtered DataFrame based on fundamental analysis:")
# print(df.head())

# file_name = "best_stocks.csv"
# path = datetime.today().strftime("%d-%m-%Y") + "/" + file_name
# df.to_csv(path)
# df.to_json(path.replace(".csv", ".json"),orient="index",indent=4)

# NEWS Anaysis


In [ ]:
import asyncio
from playwright.async_api import async_playwright
import json
import nest_asyncio

# Apply nest_asyncio for Jupyter/Colab environments
nest_asyncio.apply()

async def extract_stock_news():
    async with async_playwright() as p:
        # Launch browser (headless=True for speed, False to watch)
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        page = await context.new_page()

        # List of stocks to process
        stock_list = best_stock_ticker

        # Dictionary to store the results
        stock_news_data = {}

        print("Starting News Extraction from Groww...")

        # 1. Open Groww Homepage
        await page.goto("https://groww.in/", timeout=60000)
        # Allow initial render
        await page.wait_for_timeout(2000)

        for ticker in stock_list:
            print(f"\nProcessing: {ticker}")

            try:
                # --- STEP 1: ACTIVATE SEARCH ---
                search_input = page.locator("input#globalSearch23")

                # If input is not visible, activate it
                if not await search_input.is_visible():
                    print("  - Activating search bar...")
                    placeholder = page.locator(".se27SeSearchMainDivPlaceholder")
                    if await placeholder.count() > 0:
                        await placeholder.click(force=True)
                    else:
                        await page.keyboard.press("Control+K")

                # --- STEP 2: TYPE TICKER ---
                await search_input.wait_for(state="visible", timeout=5000)

                print(f"  - Typing '{ticker}'...")
                await search_input.fill("")
                await search_input.type(ticker, delay=100)


                # --- STEP 3: WAIT FOR SPECIFIC HTML SUGGESTION ---
                print("  - Waiting for suggestions...")

                # UPDATED SELECTOR based on your HTML snippet:
                # Target the div with class 'se27SeSuggestion'
                suggestion = page.locator(".se27SeSuggestion").first

                await suggestion.wait_for(state="visible", timeout=5000)


                print("  - Clicking first suggestion...")
                await suggestion.click()


                # --- STEP 4: HANDLE NAVIGATION ---
                # Wait for the URL to change to a stock page
                try:
                    await page.wait_for_url("**/stocks/**", timeout=15000)
                except Exception:
                    raise Exception("Clicking suggestion did not navigate to a stock page.")

                # --- STEP 5: GO TO MARKET NEWS ---
                current_url = page.url
                if current_url.endswith("/"):
                    current_url = current_url[:-1]

                news_url = f"{current_url}/market-news"
                print(f"  - Navigating to News: {news_url}")

                await page.goto(news_url, timeout=30000)

                # --- STEP 6: EXTRACT DATA ---
                news_container_selector = ".stockNews_newsRow__BC7Ia"

                try:
                    await page.wait_for_selector(news_container_selector, timeout=10000)
                except:
                    print(f"  - No news found for {ticker}")
                    stock_news_data[ticker] = []
                    await page.goto("https://groww.in/", timeout=30000)
                    continue

                news_items = page.locator(news_container_selector)
                count = await news_items.count()
                limit = min(count, 2)

                stock_news = []
                for i in range(limit):
                    item = news_items.nth(i)

                    # Extract Time
                    header_el = item.locator(".mnc671BoxHeaderText")
                    full_header = await header_el.inner_text() if await header_el.count() > 0 else ""
                    time_text = full_header.split(".")[-1].strip() if "." in full_header else full_header

                    # Extract News Text
                    body_el = item.locator(".mnc671BoxItemTitle")
                    news_text = await body_el.inner_text() if await body_el.count() > 0 else ""

                    # Extract News URL (Link)
                    link_el = item.locator("a")
                    news_link = await link_el.get_attribute("href") if await link_el.count() > 0 else ""

                    stock_news.append({
                        "time": time_text,
                        "news": news_text.replace("\n", " ").strip(),
                        "url": news_link
                    })

                stock_news_data[ticker] = stock_news
                print(f"  - Extracted {len(stock_news)} news items.")

                # Reset to home for the next loop
                await page.goto("https://groww.in/", timeout=30000)

            except Exception as e:
                print(f"  - Error processing {ticker}: {e}")
                # Save error screenshot
                await page.screenshot(path=f"debug_{ticker}_ERROR.png")
                stock_news_data[ticker] = [{"error": str(e)}]
                # Try to recover by going home
                await page.goto("https://groww.in/", timeout=30000)

        # Output Final JSON
        print("\n--- Final Extracted Data ---")
        print(json.dumps(stock_news_data, indent=4))

        await browser.close()
        return stock_news_data

if __name__ == "__main__":
    asyncio.run(extract_stock_news())

Starting News Extraction from Groww...

Processing: JINDALSAW
  - Activating search bar...
  - Typing 'JINDALSAW'...
  - Waiting for suggestions...
  - Clicking first suggestion...
  - Navigating to News: https://groww.in/stocks/jindal-saw-ltd/market-news
  - Extracted 2 news items.

Processing: ADVANIHOTR
  - Activating search bar...
  - Typing 'ADVANIHOTR'...
  - Waiting for suggestions...
  - Clicking first suggestion...
  - Navigating to News: https://groww.in/stocks/advani-hotels-resorts-india-ltd/market-news
  - Extracted 2 news items.

Processing: TECHM
  - Activating search bar...
  - Typing 'TECHM'...
  - Waiting for suggestions...
  - Clicking first suggestion...
  - Navigating to News: https://groww.in/stocks/advani-hotels-resorts-india-ltd/market-news
  - Extracted 2 news items.

--- Final Extracted Data ---
{
    "JINDALSAW": [
        {
            "time": "3 days ago",
            "news": "Jindal Saw's Q3 EBITDA declined to \u20b96B from \u20b99.4B YoY. Q3 EBITDA margin 

In [ ]:
import asyncio
from playwright.async_api import async_playwright
import pandas as pd
import nest_asyncio

# Apply nest_asyncio to allow nested event loops (essential for Colab/Jupyter)
nest_asyncio.apply()

async def get_single_stock_news_async(ticker):
    """
    Asynchronous function to fetch news for a SINGLE ticker.
    Returns a list of dictionaries (news items).
    """
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        page = await context.new_page()

        news_data = []

        try:
            # 1. Open Groww
            await page.goto("https://groww.in/", timeout=60000)

            # 2. Activate Search
            search_input = page.locator("input#globalSearch23")
            if not await search_input.is_visible():
                placeholder = page.locator(".se27SeSearchMainDivPlaceholder")
                if await placeholder.count() > 0:
                    await placeholder.click(force=True)
                else:
                    await page.keyboard.press("Control+K")

            # 3. Type Ticker
            await search_input.wait_for(state="visible", timeout=5000)
            await search_input.fill("")
            await search_input.type(ticker, delay=100)

            # 4. Click Suggestion
            suggestion = page.locator(".se27SeSuggestion").first
            await suggestion.wait_for(state="visible", timeout=5000)
            await suggestion.click()

            # 5. Wait for Navigation
            await page.wait_for_url("**/stocks/**", timeout=15000)

            # 6. Go to Market News
            current_url = page.url
            if current_url.endswith("/"):
                current_url = current_url[:-1]
            await page.goto(f"{current_url}/market-news", timeout=30000)

            # 7. Extract News
            news_container = ".stockNews_newsRow__BC7Ia"
            try:
                await page.wait_for_selector(news_container, timeout=10000)
                news_items = page.locator(news_container)
                count = await news_items.count()
                limit = min(count, 2)

                for i in range(limit):
                    item = news_items.nth(i)

                    header_el = item.locator(".mnc671BoxHeaderText")
                    full_header = await header_el.inner_text() if await header_el.count() > 0 else ""
                    time_text = full_header.split(".")[-1].strip() if "." in full_header else full_header

                    body_el = item.locator(".mnc671BoxItemTitle")
                    news_text = await body_el.inner_text() if await body_el.count() > 0 else ""

                    link_el = item.locator("a")
                    news_link = await link_el.get_attribute("href") if await link_el.count() > 0 else ""

                    news_data.append({
                        "time": time_text,
                        "news": news_text.replace("\n", " ").strip(),
                        "url": news_link
                    })
            except:
                news_data = [{"error": "No news section found"}]

        except Exception as e:
            news_data = [{"error": str(e)}]

        await browser.close()
        return news_data

def get_stock_news_sync(ticker):
    """
    Wrapper function to make the async function compatible with pandas .apply()
    """
    return asyncio.run(get_single_stock_news_async(ticker))


print("Starting extraction... This may take time as it processes row by row.")

# 2. Apply the function
# This runs the browser for EACH row sequentially.
# df["news_results"] = df["Symbol"].apply(get_stock_news_sync)

# 3. View Results
# print(df)

# To see full news for the first row:
# print(df.iloc[0]["news_results"])

Starting extraction... This may take time as it processes row by row.


In [ ]:
# Filter df to include only stocks in best_stock_ticker
df = df[df['Symbol'].isin(best_stock_ticker)]
df['news_results'] = df['Symbol'].apply(get_stock_news_sync)
# Display the filtered DataFrame (optional)
print("Filtered DataFrame based on fundamental analysis:")
print(df.head())

file_name = "best_stocks.csv"
path = datetime.today().strftime("%d-%m-%Y") + "/" + file_name
df.to_csv(path)
df.to_json(path.replace(".csv", ".json"),orient="index",indent=4)

Filtered DataFrame based on fundamental analysis:
   Sr.                               Stock Name      Symbol      Links  \
0    2                       Jindal Saw Limited   JINDALSAW  P&F | F.A   
2    4  Advani Hotels & Resorts (india) Limited  ADVANIHOTR  P&F | F.A   
6   21                    Tech Mahindra Limited       TECHM  P&F | F.A   

    % Chg    Price        Volume  trend  \
0  15.94%   179.29  12,24,65,614      1   
2   9.93%    60.36      4,16,763      1   
6   2.86%  1718.30   1,06,21,562      1   

                            pattern_BE_MS_3WS_ES_3BC  \
0  {'2025-12-26': [np.float64(-100.0), np.float64...   
2  {'2025-12-24': [np.float64(-100.0), np.float64...   
6  {'2026-01-07': [np.float64(100.0), np.float64(...   

                                           technical  \
0  {'close': np.float64(179.2899932861328), 'high...   
2  {'close': np.float64(60.36000061035156), 'high...   
6  {'close': np.float64(1718.300048828125), 'high...   

                              

In [ ]:
import asyncio
from playwright.async_api import async_playwright
import json
import nest_asyncio

nest_asyncio.apply()

async def extract_pulse_news():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        url = "https://pulse.zerodha.com/"
        print(f"Navigating to {url}...")
        await page.goto(url)

        # --- 1. DYNAMIC KEYWORD EXTRACTION ---
        print("Extracting dynamic keywords from Word Cloud...")

        # Always include these fixed categories
        active_categories = {"nifty", "ipo"}

        # Locate the word cloud links
        word_cloud_items = page.locator("#wordcloud a.word")
        count_words = await word_cloud_items.count()

        for i in range(count_words):
            el = word_cloud_items.nth(i)

            # Get the style attribute to check font size
            style_attr = await el.get_attribute("style") or ""
            # Get the clean word from data-word attribute
            word_text = await el.get_attribute("data-word")

            if word_text:
                # Logic: Check if it is a "Big Text" (32px)
                if "32px" in style_attr:
                    active_categories.add(word_text.lower())

        print(f"Active Categories: {list(active_categories)}")

        # --- 2. INITIALIZE DICTIONARY ---
        # Create a dictionary entry for every active category + a general fallback
        categorized_news = {cat: [] for cat in active_categories}
        categorized_news["general"] = []

        # --- 3. NEWS EXTRACTION ---
        news_items = page.locator("li.box.item")
        count_news = await news_items.count()
        print(f"Found {count_news} news items. Processing...")

        for i in range(count_news):
            item = news_items.nth(i)

            # Extract basic details
            headline_el = item.locator("h2 a")
            if await headline_el.count() > 0:
                headline_text = await headline_el.inner_text()
                link = await headline_el.get_attribute("href")

                summary_el = item.locator(".desc")
                summary_text = await summary_el.inner_text() if await summary_el.count() > 0 else ""

                meta_el = item.locator(".meta")
                meta_text = await meta_el.inner_text() if await meta_el.count() > 0 else ""

                news_object = {
                    "headline": headline_text.strip(),
                    "summary": summary_text.strip(),
                    "meta": meta_text.strip(),
                    "url": link
                }

                # --- 4. CATEGORIZATION LOGIC ---
                # Search for category keywords in both headline and summary
                full_text = (headline_text + " " + summary_text).lower()
                assigned = False

                # Check against our dynamic list of categories
                for category in active_categories:
                    if category in full_text:
                        categorized_news[category].append(news_object)
                        assigned = True
                        break # Stop after first match to keep it in one primary bucket

                # Fallback if no keywords matched
                if not assigned:
                    categorized_news["general"].append(news_object)

        # Output the result
        print(json.dumps(categorized_news, indent=4))

        await browser.close()
        return categorized_news

top_news = {}
if __name__ == "__main__":
     top_news = asyncio.run(extract_pulse_news())

print(top_news)

Navigating to https://pulse.zerodha.com/...
Extracting dynamic keywords from Word Cloud...
Active Categories: ['ipo', 'price', 'nifty']
Found 114 news items. Processing...
{
    "ipo": [
        {
            "headline": "Brookfield-backed Indian energy firm Clean Max said to plan reduced IPO",
            "summary": "Clean Max Enviro Energy Solutions Ltd., a renewable energy company backed by Brookfield Corp., is preparing to launch its initial public offering in February with a reduced deal size, according to people familiar with the matter.",
            "meta": "",
            "url": "https://economictimes.indiatimes.com/markets/ipos/fpos/brookfield-backed-indian-energy-firm-clean-max-said-to-plan-reduced-ipo/articleshow/126675865.cms"
        },
        {
            "headline": "Jio IPO update: Reliance awaits government notification before filing DRHP for mother of all IPOs",
            "summary": "Reliance Industries is moving closer to listing Jio Platforms, with the group aw

# Sending data


In [ ]:
import pandas as pd
import ast
import os
from datetime import datetime
import re

# 1. Setup paths
today_folder = datetime.today().strftime("%d-%m-%Y")
file_name = "best_stocks.csv"
path = os.path.join(today_folder, file_name)

if not os.path.exists(path):
    print(f"Error: The file '{path}' was not found.")
else:
    df = pd.read_csv(path)

    # --- Data Cleaning ---
    def safe_literal_eval_numpy_strings(x):
        if not isinstance(x, str) or pd.isna(x):
            if isinstance(x, dict): return x
            return {}
        try:
            cleaned_str = re.sub(r'np\.float64\((.*?)\)', r'\1', x)
            return ast.literal_eval(cleaned_str)
        except:
            return {}

    columns_to_fix = ['pattern_BE_MS_3WS_ES_3BC', 'technical', 'fundametal_resullt']
    for col in columns_to_fix:
        if col in df.columns:
            df[col] = df[col].apply(safe_literal_eval_numpy_strings)

    # --- COVER PAGE GENERATOR (NEW) ---
    def generate_cover_page():
        # Try to find a Nifty chart, else use the first stock's chart as a cover image
        nifty_chart_path = os.path.abspath(os.path.join(today_folder, "ScreenShot", "nifty.png"))
        if not os.path.exists(nifty_chart_path):
             # Fallback to the first available stock if Nifty chart isn't there
            first_symbol = df.iloc[0]['Symbol'] if not df.empty else "SANDHAR"
            nifty_chart_path = os.path.abspath(os.path.join(today_folder, "ScreenShot", f"{first_symbol}.png"))

        date_str = datetime.now().strftime('%d %B, %Y')

        return f"""
        <div class="page-container cover-page">
            <div class="cover-header">
                <div class="company-logo">
                    <span class="logo-icon">📈</span> MARKET<span style="color:#1a73e8">PULSE</span>
                </div>
                <div class="report-date">{date_str}</div>
            </div>

            <div class="cover-content">
                <h1 class="cover-title">NIFTY MARKET<br>ANALYSIS</h1>
                <div class="cover-subtitle">Daily Technical & Fundamental Report</div>

                <div class="cover-chart-box">
                    <div class="section-header">MARKET SNAPSHOT</div>
                    <div class="crop-viewport" style="height: 500px; border: 4px solid #333;">
                        <img src="file:///{nifty_chart_path}" class="chart-img" alt="Market Chart">
                    </div>
                </div>

                <div class="market-summary-box">
                    <h3>Market Commentary</h3>
                    <p>This report provides a comprehensive analysis of technically strong stocks, breakout candidates, and fundamentally sound companies for the upcoming trading sessions.</p>
                </div>
            </div>

            <div class="cover-footer">
                Generated by Auto-Analyst Bot
            </div>
        </div>
        """

    # --- STOCK PAGE GENERATOR ---
    def generate_stock_html(stock_index, stock_data):
        technical_data = stock_data.get('technical', {})
        fundamental = stock_data.get('fundametal_resullt', {})
        symbol = stock_data.get('Symbol', 'N/A')
        abs_screenshot_path = os.path.abspath(os.path.join(today_folder, "ScreenShot", f"{symbol}.png"))

        # Technical HTML
        price_keys = ['open', 'high', 'low', 'close', 'volume']
        price_html = '<div class="price-grid">'
        for key in price_keys:
            val = technical_data.get(key, 0)
            disp = f"{val:,.0f}" if key == 'volume' else f"{val:.2f}"
            price_html += f'<div class="price-item"><span class="price-label">{key.upper()}</span><span class="price-value">{disp}</span></div>'
        price_html += '</div>'

        indicators_html = '<div class="indicators-grid">'
        for indicator, value in technical_data.items():
            if indicator in price_keys: continue
            if isinstance(value, (int, float)):
                indicator_class = "overbought" if 'RSI' in indicator and value > 70 else "oversold" if 'RSI' in indicator and value < 30 else ""
                indicators_html += f'<div class="tech-item {indicator_class}"><div class="label">{indicator}</div><div class="value">{value:.2f}</div></div>'
        indicators_html += '</div>'

        # Fundamental HTML
        scores_data = fundamental.get('scores', {})
        if not scores_data and 'details' in fundamental:
             scores_data = {k: v.get('score_pct', 0) for k,v in fundamental['details'].items()}
        scores_html = "".join([f'<div class="score-card"><div class="name">{k}</div><div class="score">{v}</div></div>' for k, v in scores_data.items()])

        rating_text = fundamental.get('rating', 'N/A')
        rating_class = "rating-good" if any(x in str(rating_text).upper() for x in ["GOOD", "EXCELLENT"]) else "rating-average"

        # News HTML
        news_items = stock_data.get('news_results', {})
        news_html = ""
        # ... (Your robust news parsing logic from previous step) ...
        if isinstance(news_items, str):
            try: news_items = ast.literal_eval(news_items)
            except: news_items = []
        if isinstance(news_items, list) and news_items:
            news_rows = ""
            count = 0
            for item in news_items:
                if count >= 3: break
                if isinstance(item, str):
                    try: item = ast.literal_eval(item)
                    except: item = {'news': item, 'time': '', 'url': '#'}
                if not isinstance(item, dict): continue

                news_rows += f"""
                <div class="news-item">
                    <div class="news-meta"><span class="news-icon">📰</span> {item.get('time', 'Recent')} <a href="{item.get('url', '#')}" target="_blank" class="news-link">Open Link ↗</a></div>
                    <div class="news-text">{item.get('news', 'No details.')}</div>
                </div>"""
                count += 1
            if news_rows:
                news_html = f'<div class="section-box"><div class="section-header">LATEST NEWS</div><div class="news-container">{news_rows}</div></div>'

        return f"""
        <div class="page-container">
            <div class="header"><h1>{stock_data.get('Stock Name', 'N/A')}</h1><div class="stock-info">{symbol} | {datetime.now().strftime('%d-%m-%Y')}</div></div>
            <div class="main-price-bar"><span class="main-price">₹{stock_data.get('Price', 'N/A')}</span><span class="pct-chg">({stock_data.get('% Chg', 'N/A')})</span><span class="vol-text">Vol: {stock_data.get('Volume', 'N/A')}</span></div>
            <div class="section-box"><div class="section-header">TECHNICAL ANALYSIS</div>{price_html}{indicators_html}</div>
            <div class="section-box"><div class="section-header">FUNDAMENTAL ANALYSIS</div><div style="margin-bottom: 5px;"><span class="rating-badge {rating_class}">{rating_text}</span> <span class="risk-label" style="font-size:10px;">Risk: <strong>{fundamental.get('risk_level', 'N/A')}</strong></span></div><div class="scores-grid">{scores_html}</div></div>
            {news_html}
            <div class="chart-container"><div class="section-header">PRICE CHART</div><div class="crop-viewport"><img src="file:///{abs_screenshot_path}" class="chart-img" alt="Chart Missing"></div></div>
        </div>
        """

    # --- MAIN EXECUTION ---
    cover_page_html = generate_cover_page()
    all_stocks_html = "".join([generate_stock_html(i+1, row) for i, row in df.iterrows()])

    # Generic Top News HTML Generator
    top_news_html = '<div class="news-dashboard"><div class="news-grid">'
    # Assuming 'top_news' dictionary is available from your previous context
    top_news = locals().get('top_news', {})
    sorted_categories = sorted(top_news.keys())
    for index, category in enumerate(sorted_categories):
        items = top_news[category]
        if not items: continue
        color_index = index % 8
        badge_class = f"badge-{color_index}"
        display_name = category.upper() if len(category) <= 3 else category.title()
        top_news_html += f'<div class="category-column"><div class="category-header"><span class="category-badge {badge_class}">{display_name}</span><h3 class="category-title">{display_name} Updates</h3></div><ul class="news-list">'
        for item in items[:5]:
            top_news_html += f'<li class="news-item"><a href="{item.get("url", "#")}" target="_blank" class="news-headline">{item.get("headline", "No Headline")}</a><p class="news-summary">{item.get("summary", "Click to read.") or "Click to read."}</p></li>'
        top_news_html += "</ul></div>"
    top_news_html += '</div></div>'

    filtered_stocks_list = locals().get('filtered_stocks', 'N/A')
    best_stocks_list = locals().get('best_stock_ticker', 'N/A')

    complete_html = f"""
<!DOCTYPE html>
<html>
<head>
    <style>
        @page {{ size: A4; margin: 0; }}
        body {{ font-family: 'Helvetica', sans-serif; margin: 0; padding: 0; background: #fff; color: #222; }}

        .page-container {{ width: 210mm; height: 296mm; padding: 10mm; box-sizing: border-box; page-break-after: always; overflow: hidden; display: flex; flex-direction: column; }}

        /* --- Cover Page Styles --- */
        .cover-page {{ justify-content: space-between; background: #fdfdfd; }}
        .cover-header {{ display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #333; padding-bottom: 20px; }}
        .company-logo {{ font-size: 24px; font-weight: bold; color: #333; display: flex; align-items: center; gap: 10px; }}
        .logo-icon {{ font-size: 30px; }}
        .report-date {{ font-size: 14px; color: #666; font-weight: 600; }}

        .cover-content {{ flex-grow: 1; display: flex; flex-direction: column; justify-content: center; align-items: center; text-align: center; }}
        .cover-title {{ font-size: 48px; font-weight: 900; line-height: 1.1; margin: 0 0 10px 0; color: #111; letter-spacing: -1px; text-transform: uppercase; }}
        .cover-subtitle {{ font-size: 18px; color: #555; margin-bottom: 40px; font-weight: 300; }}

        .cover-chart-box {{ width: 100%; margin-bottom: 30px; }}
        .market-summary-box {{ background: #f8f9fa; border-left: 5px solid #1a73e8; padding: 20px; text-align: left; width: 100%; box-sizing: border-box; }}
        .market-summary-box h3 {{ margin: 0 0 10px 0; color: #1a73e8; font-size: 16px; }}
        .market-summary-box p {{ margin: 0; font-size: 12px; line-height: 1.5; color: #444; }}

        .cover-footer {{ text-align: center; font-size: 10px; color: #999; border-top: 1px solid #eee; padding-top: 20px; }}

        /* --- Standard Styles --- */
        .header {{ background: #1a73e8; color: white; padding: 8px; text-align: center; border-radius: 4px; flex-shrink: 0; }}
        .header h1 {{ margin: 0; font-size: 16px; }}
        .stock-info {{ font-size: 10px; opacity: 0.9; }}
        .main-price-bar {{ text-align: center; padding: 8px 0; border-bottom: 1px solid #eee; flex-shrink: 0; }}
        .main-price {{ font-size: 20px; font-weight: bold; }}
        .pct-chg {{ font-size: 14px; color: #28a745; margin-left: 5px; }}
        .vol-text {{ color: #666; font-size: 10px; margin-left: 10px; }}
        .section-box {{ margin-top: 8px; flex-shrink: 0; }}
        .section-header {{ font-size: 11px; font-weight: bold; color: #1a73e8; border-left: 3px solid #1a73e8; padding-left: 6px; margin-bottom: 5px; }}
        .price-grid {{ display: flex; justify-content: space-between; margin-bottom: 5px; background: #f8f9fa; padding: 4px; border-radius: 4px; }}
        .price-item {{ text-align: center; flex: 1; }}
        .price-label {{ font-size: 8px; color: #777; display: block; }}
        .price-value {{ font-size: 10px; font-weight: bold; }}
        .indicators-grid, .scores-grid {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 3px; }}
        .scores-grid {{ grid-template-columns: repeat(3, 1fr); }}
        .tech-item, .score-card {{ border: 1px solid #eee; padding: 3px; text-align: center; border-radius: 3px; }}
        .label, .name {{ font-size: 7px; color: #888; display: block; }}
        .value, .score {{ font-size: 9px; font-weight: bold; }}
        .rating-badge {{ padding: 2px 6px; border-radius: 8px; color: white; font-size: 9px; font-weight: bold; }}
        .rating-good {{ background: #28a745; }}
        .rating-average {{ background: #ffc107; color: #000; }}
        .news-container {{ display: flex; flex-direction: column; gap: 4px; }}
        .news-item {{ background: #fdfdfd; border-left: 2px solid #ddd; padding: 4px 8px; }}
        .news-meta {{ font-size: 9px; color: #666; margin-bottom: 2px; display: flex; justify-content: space-between; }}
        .news-link {{ color: #1a73e8; text-decoration: none; font-size: 8px; font-weight: bold; }}
        .news-text {{ font-size: 10px; color: #333; line-height: 1.3; text-align: justify; }}
        .chart-container {{ margin-top: 8px; flex-grow: 1; min-height: 0; display: flex; flex-direction: column; }}
        .crop-viewport {{ flex-grow: 1; width: 100%; overflow: hidden; border: 1px solid #ddd; position: relative; background: #fdfdfd; }}
        .chart-img {{ width: 100%; position: absolute; top: -50px; left: 0; }}
        .overbought {{ color: #dc3545; }}
        .oversold {{ color: #28a745; }}

        /* Summary & News Dashboard Styles */
        .summary-container {{ width: 90%; border: 2px solid #1a73e8; border-radius: 12px; padding: 30px; background: white; margin: 20px auto; }}
        .summary-header {{ display: flex; align-items: center; gap: 15px; border-bottom: 1px solid #eee; padding-bottom: 15px; margin-bottom: 20px; }}
        .news-dashboard {{ font-family: -apple-system, sans-serif; color: #333; padding: 20px; background-color: #f4f6f9; }}
        .news-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(320px, 1fr)); gap: 20px; align-items: start; }}
        .category-column {{ background: #fff; border-radius: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.08); overflow: hidden; border: 1px solid #e1e4e8; display: flex; flex-direction: column; }}
        .category-header {{ padding: 12px 15px; border-bottom: 2px solid #f0f0f0; display: flex; align-items: center; background: #ffffff; }}
        .category-badge {{ text-transform: uppercase; font-size: 11px; font-weight: 800; color: #fff; padding: 4px 8px; border-radius: 4px; margin-right: 10px; }}
        .badge-0 {{ background-color: #007bff; }} .badge-1 {{ background-color: #28a745; }} .badge-2 {{ background-color: #6f42c1; }}
        .badge-3 {{ background-color: #fd7e14; }} .badge-4 {{ background-color: #e83e8c; }} .badge-5 {{ background-color: #17a2b8; }}
        .badge-6 {{ background-color: #d63384; }} .badge-7 {{ background-color: #6610f2; }}
        .category-title {{ margin: 0; font-size: 15px; font-weight: 700; color: #2c3e50; text-transform: capitalize; }}
        .news-list {{ list-style: none; padding: 0; margin: 0; }}
        .news-item {{ padding: 15px; border-bottom: 1px solid #f1f1f1; }}
        .news-item:last-child {{ border-bottom: none; }}
        .news-headline {{ display: block; font-size: 13px; font-weight: 600; color: #2c3e50; margin-bottom: 5px; text-decoration: none; }}
        .news-summary {{ font-size: 12px; color: #666; line-height: 1.5; margin: 0; display: -webkit-box; -webkit-line-clamp: 3; -webkit-box-orient: vertical; overflow: hidden; }}
    </style>
</head>
<body>
    {cover_page_html}
    {all_stocks_html}

    <div class="page-container" style="justify-content: center; height: auto; min-height: 296mm;">
        <div class="summary-container"><div class="summary-header"><span class="icon">⚡</span><h2>TECHNICALLY STRONG STOCKS</h2></div><div> {filtered_stocks_list} </div></div>
        <div class="summary-container"><div class="summary-header"><span class="icon">⚡</span><h2>TECHNICALLY AND FUNDAMENTALLY STRONG STOCKS</h2></div><div> {best_stocks_list} </div></div>
        {top_news_html}
    </div>
</body>
</html>
"""
    output_html_path = os.path.join(today_folder, "stock_analysis_report.html")
    with open(output_html_path, "w", encoding="utf-8") as f:
        f.write(complete_html)
    print(f"Success! Saved to {output_html_path}")

Success! Saved to 19-01-2026/stock_analysis_report.html


In [ ]:
from playwright.async_api import async_playwright
import asyncio
import os

async def convert_html_to_pdf(html_file_path, output_pdf_path):
    async with async_playwright() as p:
        # Launch headless browser
        browser = await p.chromium.launch()
        page = await browser.new_page()

        # Load the local HTML file
        # Use an absolute path for the file URL
        absolute_html_path = os.path.abspath(html_file_path)
        await page.goto(f"file:///{absolute_html_path}")

        # Wait for images to load before printing
        await page.wait_for_load_state("networkidle")

        # Emulate print media for A4 size
        await page.pdf(
            path=output_pdf_path,
            format="A4",
            print_background=True, # Critical for CSS colors and backgrounds
            margin={"top": "1cm", "right": "1cm", "bottom": "1cm", "left": "1cm"},
            display_header_footer=False
        )

        await browser.close()
        print(f"PDF successfully created at: {output_pdf_path}")

# --- Execution ---
html_path = os.path.join(today_folder, "stock_analysis_report.html")
pdf_path = os.path.join(today_folder, "Final_Stock_Analysis.pdf")

# Generate the PDF
await convert_html_to_pdf(html_path, pdf_path)

PDF successfully created at: 19-01-2026/Final_Stock_Analysis.pdf


In [ ]:
import os
import smtplib
from email.message import EmailMessage

# 1. Setup paths (Matching your previous logic)
today_folder = datetime.today().strftime("%d-%m-%Y")
pdf_path = os.path.join(today_folder, "Final_Stock_Analysis.pdf")

# 2. Extract Tickers for Response
# Assuming df is your filtered dataframe from previous steps
best_stock_tickers = df['Symbol'].tolist()

# --- EMAIL SENDING LOGIC ---
SMTP_HOST = "smtp.gmail.com"
SMTP_PORT = 587
EMAIL = "knpatel0707@gmail.com"
APP_PASSWORD = "ules habb jtdm jvni"

msg = EmailMessage()
msg["Subject"] = f"Best Stocks for Tomorrow - {today_folder}"
msg["From"] = EMAIL
msg["To"] = "kevalnpatel070@gmail.com"

# Plain text body
body_content = f"""Hello,

Please find the attached PDF for today's best stock analysis.

Best Stock Tickers:
{', '.join(best_stock_tickers)}

Technically Strong Stocks:
{', '.join(filtered_stocks_list)}

Best regards,
Stock Scanner Bot"""

msg.set_content(body_content)

# 3. Attach the PDF file
if os.path.exists(pdf_path):
    with open(pdf_path, "rb") as f:
        file_data = f.read()
        file_name = os.path.basename(pdf_path)
        msg.add_attachment(
            file_data,
            maintype="application",
            subtype="pdf",
            filename=file_name
        )

    # Send the Email
    try:
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as smtp:
            smtp.ehlo()
            smtp.starttls()
            smtp.login(EMAIL, APP_PASSWORD)
            smtp.send_message(msg)
        print(f"Email sent successfully with attachment: {file_name}")
    except Exception as e:
        print(f"Failed to send email: {e}")
else:
    print(f"Error: PDF file not found at {pdf_path}. Email not sent.")

# 4. Response Output
print("\n--- Summary ---")
print(f"Best Stock Tickers: {best_stock_tickers}")
print(f"Filtered (Technically Strong): {filtered_stocks_list}")

# EXTRA


In [ ]:
# #final version 1 without nifty page
# import pandas as pd
# import ast
# import os
# from datetime import datetime
# import re

# # 1. Setup paths
# today_folder = datetime.today().strftime("%d-%m-%Y")
# file_name = "best_stocks.csv"
# path = os.path.join(today_folder, file_name)

# if not os.path.exists(path):
#     print(f"Error: The file '{path}' was not found.")
# else:
#     df = pd.read_csv(path)

#     # --- Data Cleaning Functions ---
#     def safe_literal_eval_numpy_strings(x):
#         if not isinstance(x, str) or pd.isna(x):
#             if isinstance(x, dict): return x
#             return {}
#         try:
#             # Remove numpy wrappers if present
#             cleaned_str = re.sub(r'np\.float64\((.*?)\)', r'\1', x)
#             return ast.literal_eval(cleaned_str)
#         except:
#             return {}

#     # Apply cleaning
#     columns_to_fix = ['pattern_BE_MS_3WS_ES_3BC', 'technical', 'fundametal_resullt']
#     for col in columns_to_fix:
#         if col in df.columns:
#             df[col] = df[col].apply(safe_literal_eval_numpy_strings)

#     # --- HTML Generation Function ---
#     def generate_stock_html(stock_index, stock_data):
#         technical_data = stock_data.get('technical', {})
#         fundamental = stock_data.get('fundametal_resullt', {})
#         symbol = stock_data.get('Symbol', 'N/A')

#         abs_screenshot_path = os.path.abspath(os.path.join(today_folder, "ScreenShot", f"{symbol}.png"))

#         # 1. Technical Section HTML
#         price_keys = ['open', 'high', 'low', 'close', 'volume']
#         price_html = '<div class="price-grid">'
#         for key in price_keys:
#             val = technical_data.get(key, 0)
#             disp = f"{val:,.0f}" if key == 'volume' else f"{val:.2f}"
#             price_html += f'<div class="price-item"><span class="price-label">{key.upper()}</span><span class="price-value">{disp}</span></div>'
#         price_html += '</div>'

#         indicators_html = '<div class="indicators-grid">'
#         for indicator, value in technical_data.items():
#             if indicator in price_keys: continue
#             if isinstance(value, (int, float)):
#                 indicator_class = "overbought" if 'RSI' in indicator and value > 70 else "oversold" if 'RSI' in indicator and value < 30 else ""
#                 indicators_html += f'<div class="tech-item {indicator_class}"><div class="label">{indicator}</div><div class="value">{value:.2f}</div></div>'
#         indicators_html += '</div>'

#         # 2. Fundamental Scores HTML
#         scores_data = fundamental.get('scores', {})
#         # Fallback if scores are missing but 'details' exists (handling user json variation)
#         if not scores_data and 'details' in fundamental:
#              # Basic extraction if structure is nested differently
#              scores_data = {k: v.get('score_pct', 0) for k,v in fundamental['details'].items()}

#         scores_html = "".join([f'<div class="score-card"><div class="name">{k}</div><div class="score">{v}</div></div>'
#                               for k, v in scores_data.items()])

#         rating_text = fundamental.get('rating', 'N/A')
#         rating_class = "rating-good" if any(x in str(rating_text).upper() for x in ["GOOD", "EXCELLENT"]) else "rating-average"

#         # 3. News Section HTML (NEW)
#         # news_items = stock_data.get('news_results', [])
#         # news_items = stock_data.get('news_results')

#         # 3. News Section HTML (Corrected & Robust)
#         news_items = stock_data.get('news_results', {})
#         news_html = ""

#         # 1. Ensure news_items is actually a list (handles case where it's a string representation)
#         if isinstance(news_items, str):
#             try:
#                 import ast
#                 news_items = ast.literal_eval(news_items)
#             except:
#                 news_items = []

#         if isinstance(news_items, list) and news_items:
#             news_rows = ""
#             count = 0

#             for item in news_items:
#                 if count >= 3: break # Limit to 3 items

#                 # 2. Defensive check: If item is a string (double-encoded JSON), try to parse it
#                 if isinstance(item, str):
#                     try:
#                         import ast
#                         item = ast.literal_eval(item)
#                     except:
#                         # If parsing fails, treat the string as the headline itself
#                         item = {'news': item, 'time': '', 'url': '#'}

#                 # 3. Ensure item is now a dictionary before calling .get()
#                 if not isinstance(item, dict):
#                     continue

#                 time_ago = item.get('time', 'Recent')
#                 headline = item.get('news', 'No details available.')
#                 url = item.get('url', '#')

#                 news_rows += f"""
#                 <div class="news-item">
#                     <div class="news-meta">
#                         <span class="news-icon">📰</span> {time_ago}
#                         <a href="{url}" target="_blank" class="news-link">Open Link ↗</a>
#                     </div>
#                     <div class="news-text">{headline}</div>
#                 </div>
#                 """
#                 count += 1

#             if news_rows:
#                 news_html = f"""
#                 <div class="section-box">
#                     <div class="section-header">LATEST NEWS</div>
#                     <div class="news-container">
#                         {news_rows}
#                     </div>
#                 </div>
#                 """

#         # 4. Construct Full Page
#         return f"""
#         <div class="page-container">
#             <div class="header">
#                 <h1>{stock_data.get('Stock Name', 'N/A')}</h1>
#                 <div class="stock-info">{symbol} | {datetime.now().strftime('%d-%m-%Y')}</div>
#             </div>

#             <div class="main-price-bar">
#                 <span class="main-price">₹{stock_data.get('Price', 'N/A')}</span>
#                 <span class="pct-chg">({stock_data.get('% Chg', 'N/A')})</span>
#                 <span class="vol-text">Vol: {stock_data.get('Volume', 'N/A')}</span>
#             </div>

#             <div class="section-box">
#                 <div class="section-header">TECHNICAL ANALYSIS</div>
#                 {price_html}
#                 {indicators_html}
#             </div>

#             <div class="section-box">
#                 <div class="section-header">FUNDAMENTAL ANALYSIS</div>
#                 <div style="margin-bottom: 5px; display:flex; justify-content:space-between; align-items:center;">
#                     <div>
#                         <span class="rating-badge {rating_class}">{rating_text}</span>
#                         <span class="risk-label" style="margin-left:10px; font-size:10px;">Risk: <strong>{fundamental.get('risk_level', 'N/A')}</strong></span>
#                     </div>
#                 </div>
#                 <div class="scores-grid">{scores_html}</div>
#             </div>


#             <div class="chart-container">
#                 <div class="section-header">PRICE CHART</div>
#                 <div class="crop-viewport">
#                     <img src="file:///{abs_screenshot_path}" class="chart-img" alt="Chart Missing">
#                 </div>
#             </div>

#             {news_html}
#         </div>
#         """

#     # Generate HTML for all rows
#     all_stocks_html = "".join([generate_stock_html(i+1, row) for i, row in df.iterrows()])

#     # Use variable names from your context for the summary section, or empty string if not defined
#     filtered_stocks_list = locals().get('filtered_stocks', 'N/A')
#     best_stocks_list = locals().get('best_stock_ticker', 'N/A')


#     # top_news_html

#     top_news_html = '<div class="news-dashboard"><div class="news-grid">'

#     # Sort categories so they appear in a consistent order (e.g., Alphabetical)
#     # You can change this to sort by number of items: key=lambda k: len(news_data[k]), reverse=True
#     sorted_categories = sorted(top_news.keys())

#     for index, category in enumerate(sorted_categories):
#         items = top_news[category]
#         if not items: continue  # Skip empty categories

#         # Cycle through the 8 generic color classes defined in CSS
#         color_index = index % 8
#         badge_class = f"badge-{color_index}"

#         # Format the Category Name (e.g., "ipo" -> "IPO", "nifty" -> "Nifty")
#         # .upper() if length <= 3 (like IPO), else .title()
#         display_name = category.upper() if len(category) <= 3 else category.title()

#         top_news_html += f"""
#         <div class="category-column">
#             <div class="category-header">
#                 <span class="category-badge {badge_class}">{display_name}</span>
#                 <h3 class="category-title">{display_name} Updates</h3>
#             </div>
#             <ul class="news-list">
#         """

#         # Loop through items (Limit to top 5)
#         for item in items[:5]:
#             headline = item.get('headline', 'No Headline')
#             summary = item.get('summary', '')
#             url = item.get('url', '#')

#             if not summary:
#                 summary = "Click to read the full story."

#             top_news_html += f"""
#                 <li class="news-item">
#                     <a href="{url}" target="_blank" class="news-headline">{headline}</a>
#                     <p class="news-summary">{summary}</p>
#                 </li>
#             """

#         top_news_html += "</ul></div>" # Close Column

#     top_news_html += '</div></div>' # Close Grid & Dashboard


#     complete_html = f"""
# <!DOCTYPE html>
# <html>
# <head>
#     <style>
#         @page {{ size: A4; margin: 0; }}
#         body {{ font-family: 'Helvetica', sans-serif; margin: 0; padding: 0; background: #fff; color: #222; }}

#         /* Main Flex Container for A4 Page */
#         .page-container {{
#             width: 210mm;
#             height: 296mm; /* Strict A4 Height */
#             padding: 10mm;
#             box-sizing: border-box;
#             page-break-after: always;
#             overflow: hidden;
#             display: flex;
#             flex-direction: column; /* Stack elements vertically */
#         }}

#         .header {{ background: #1a73e8; color: white; padding: 8px; text-align: center; border-radius: 4px; flex-shrink: 0; }}
#         .header h1 {{ margin: 0; font-size: 16px; }}
#         .stock-info {{ font-size: 10px; opacity: 0.9; }}

#         .main-price-bar {{ text-align: center; padding: 8px 0; border-bottom: 1px solid #eee; flex-shrink: 0; }}
#         .main-price {{ font-size: 20px; font-weight: bold; }}
#         .pct-chg {{ font-size: 14px; color: #28a745; margin-left: 5px; }}
#         .vol-text {{ color: #666; font-size: 10px; margin-left: 10px; }}

#         .section-box {{ margin-top: 8px; flex-shrink: 0; }}
#         .section-header {{ font-size: 11px; font-weight: bold; color: #1a73e8; border-left: 3px solid #1a73e8; padding-left: 6px; margin-bottom: 5px; }}

#         .price-grid {{ display: flex; justify-content: space-between; margin-bottom: 5px; background: #f8f9fa; padding: 4px; border-radius: 4px; }}
#         .price-item {{ text-align: center; flex: 1; }}
#         .price-label {{ font-size: 8px; color: #777; display: block; }}
#         .price-value {{ font-size: 10px; font-weight: bold; }}

#         .indicators-grid, .scores-grid {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 3px; }}
#         .scores-grid {{ grid-template-columns: repeat(3, 1fr); }}

#         .tech-item, .score-card {{ border: 1px solid #eee; padding: 3px; text-align: center; border-radius: 3px; }}
#         .label, .name {{ font-size: 7px; color: #888; display: block; }}
#         .value, .score {{ font-size: 9px; font-weight: bold; }}

#         .rating-badge {{ padding: 2px 6px; border-radius: 8px; color: white; font-size: 9px; font-weight: bold; }}
#         .rating-good {{ background: #28a745; }}
#         .rating-average {{ background: #ffc107; color: #000; }}

#         /* --- News Section Styles --- */
#         .news-container {{ display: flex; flex-direction: column; gap: 4px; }}
#         .news-item {{ background: #fdfdfd; border-left: 2px solid #ddd; padding: 4px 8px; }}
#         .news-meta {{ font-size: 9px; color: #666; margin-bottom: 2px; display: flex; justify-content: space-between; }}
#         .news-link {{ color: #1a73e8; text-decoration: none; font-size: 8px; font-weight: bold; }}
#         .news-text {{ font-size: 10px; color: #333; line-height: 1.3; text-align: justify; }}

#         /* --- Chart Section (Fills Remaining Space) --- */
#         .chart-container {{
#             margin-top: 8px;
#             flex: 0 1 400px;
#             min-height: 0;
#             display: flex;
#             flex-direction: column;
#         }}

#         .crop-viewport {{
#             flex-grow: 1;      /* Fill the parent chart-container */
#             width: 100%;
#             overflow: hidden;  /* Crop whatever overflows */
#             border: 1px solid #ddd;
#             position: relative;
#             background: #fdfdfd;
#         }}
#         .chart-img {{
#             width: 100%;
#             position: absolute;
#             top: -50px; /* Hide top nav of website screenshot */
#             left: 0;
#         }}

#         .overbought {{ color: #dc3545; }}
#         .oversold {{ color: #28a745; }}

#         /* Summary Pages */
#         .summary-container {{
#             width: 90%;
#             border: 2px solid #1a73e8;
#             border-radius: 12px;
#             padding: 30px;
#             background: white;
#             margin: 20px auto;
#         }}


#         /* top news
#         /* --- News Dashboard Container --- */
#         .news-dashboard {{
#             font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
#             color: #333;
#             padding: 20px;
#             background-color: #f4f6f9;
#         }}

#         /* --- Responsive Grid Layout --- */
#         .news-grid {{
#             display: grid;
#             grid-template-columns: repeat(auto-fit, minmax(320px, 1fr)); /* Auto-sizing columns */
#             gap: 20px;
#             align-items: start;
#         }}

#         /* --- Category Card --- */
#         .category-column {{
#             background: #fff;
#             border-radius: 8px;
#             box-shadow: 0 2px 5px rgba(0,0,0,0.08);
#             overflow: hidden;
#             border: 1px solid #e1e4e8;
#             display: flex;
#             flex-direction: column;
#         }}

#         /* --- Header --- */
#         .category-header {{
#             padding: 12px 15px;
#             border-bottom: 2px solid #f0f0f0;
#             display: flex;
#             align-items: center;
#             background: #ffffff;
#         }}

#         .category-badge {{
#             text-transform: uppercase;
#             font-size: 11px;
#             font-weight: 800;
#             color: #fff;
#             padding: 4px 8px;
#             border-radius: 4px;
#             margin-right: 10px;
#             letter-spacing: 0.5px;
#         }}

#         /* --- Generic Color Palette (Cycles automatically) --- */
#         .badge-0 {{ background-color: #007bff; }} /* Blue */
#         .badge-1 {{ background-color: #28a745; }} /* Green */
#         .badge-2 {{ background-color: #6f42c1; }} /* Purple */
#         .badge-3 {{ background-color: #fd7e14; }} /* Orange */
#         .badge-4 {{ background-color: #e83e8c; }} /* Pink */
#         .badge-5 {{ background-color: #17a2b8; }} /* Cyan */
#         .badge-6 {{ background-color: #d63384; }} /* Magenta */
#         .badge-7 {{ background-color: #6610f2; }} /* Indigo */

#         .category-title {{
#             margin: 0;
#             font-size: 15px;
#             font-weight: 700;
#             color: #2c3e50;
#             text-transform: capitalize; /* Ensures "ipo" becomes "Ipo" */
#         }}

#         /* --- News List Items --- */
#         .news-list {{
#             list-style: none;
#             padding: 0;
#             margin: 0;
#         }}

#         .news-item {{
#             padding: 15px;
#             border-bottom: 1px solid #f1f1f1;
#             transition: background-color 0.2s;
#         }}

#         .news-item:last-child {{ border-bottom: none; }}

#         .news-item:hover {{ background-color: #f9fbfd; }}

#         .news-headline {{
#             display: block;
#             font-size: 13px;
#             font-weight: 600;
#             color: #2c3e50;
#             margin-bottom: 5px;
#             line-height: 1.4;
#             text-decoration: none;
#         }}

#         .news-headline:hover {{
#             color: #007bff;
#             text-decoration: underline;
#         }}

#         .news-summary {{
#             font-size: 12px;
#             color: #666;
#             line-height: 1.5;
#             margin: 0;
#             /* Truncate text after 3 lines */
#             display: -webkit-box;
#             -webkit-line-clamp: 3;
#             -webkit-box-orient: vertical;
#             overflow: hidden;
#         }}
#         .summary-header {{ display: flex; align-items: center; gap: 15px; border-bottom: 1px solid #eee; padding-bottom: 15px; margin-bottom: 20px; }}
#     </style>
# </head>
# <body>

#     {all_stocks_html}

#     <div class="page-container" style="justify-content: center; height: auto; min-height: 296mm;">
#         <div class="summary-container">
#             <div class="summary-header">
#                 <span class="icon">⚡</span>
#                 <h2>TECHNICALLY STRONG STOCKS</h2>
#             </div>
#             <div> {filtered_stocks_list} </div>
#         </div>
#         <div class="summary-container">
#             <div class="summary-header">
#                 <span class="icon">⚡</span>
#                 <h2>TECHNICALLY AND FUNDAMENTALLY STRONG STOCKS</h2>
#             </div>
#             <div> {best_stocks_list} </div>
#         </div>

#         {top_news_html}
#     </div>
# </body>
# </html>
# """
#     output_html_path = os.path.join(today_folder, "stock_analysis_report.html")
#     with open(output_html_path, "w", encoding="utf-8") as f:
#         f.write(complete_html)
#     print(f"Success! Saved to {output_html_path}")

In [ ]:
# best_stock_ticker

['SANDHAR',
 'CARRARO',
 'ROLEXRINGS',
 'AUTOAXLES',
 'MANORAMA',
 'VOLTAS',
 'RELIABLE']

In [ ]:
# import pandas as pd
# import json
# import ast
# import os
# from datetime import datetime
# import re

# # 1. Setup paths
# today_folder = datetime.today().strftime("%d-%m-%Y")
# file_name = "best_stocks.csv"
# path = os.path.join(today_folder, file_name)

# if not os.path.exists(path):
#     print(f"Error: The file '{path}' was not found.")
# else:
#     df = pd.read_csv(path)

#     def safe_literal_eval_numpy_strings(x):
#         if not isinstance(x, str) or pd.isna(x):
#             if isinstance(x, dict): return x
#             return {}
#         try:
#             cleaned_str = re.sub(r'np\.float64\((.*?)\)', r'\1', x)
#             return ast.literal_eval(cleaned_str)
#         except:
#             return {}

#     df['pattern_BE_MS_3WS_ES_3BC'] = df['pattern_BE_MS_3WS_ES_3BC'].apply(safe_literal_eval_numpy_strings)
#     df['technical'] = df['technical'].apply(safe_literal_eval_numpy_strings)
#     df['fundametal_resullt'] = df['fundametal_resullt'].apply(safe_literal_eval_numpy_strings)

#     def generate_stock_html(stock_index, stock_data):
#         technical_data = stock_data['technical']
#         fundamental = stock_data['fundametal_resullt']
#         symbol = stock_data.get('Symbol', 'N/A')

#         abs_screenshot_path = os.path.abspath(os.path.join(today_folder, "ScreenShot", f"{symbol}.png"))

#         # --- Technical Section ---
#         price_keys = ['open', 'high', 'low', 'close', 'volume']
#         price_html = '<div class="price-grid">'
#         for key in price_keys:
#             val = technical_data.get(key, 0)
#             disp = f"{val:,.0f}" if key == 'volume' else f"{val:.2f}"
#             price_html += f'<div class="price-item"><span class="price-label">{key.upper()}</span><span class="price-value">{disp}</span></div>'
#         price_html += '</div>'

#         indicators_html = '<div class="indicators-grid">'
#         for indicator, value in technical_data.items():
#             if indicator in price_keys: continue
#             if isinstance(value, (int, float)):
#                 indicator_class = "overbought" if 'RSI' in indicator and value > 70 else "oversold" if 'RSI' in indicator and value < 30 else ""
#                 indicators_html += f'<div class="tech-item {indicator_class}"><div class="label">{indicator}</div><div class="value">{value:.2f}</div></div>'
#         indicators_html += '</div>'

#         # --- Fundamental Scores ---
#         scores_html = "".join([f'<div class="score-card"><div class="name">{k}</div><div class="score">{v}</div></div>'
#                               for k, v in fundamental.get('scores', {}).items()])

#         rating_text = fundamental.get('rating', 'N/A')
#         rating_class = "rating-good" if any(x in rating_text.upper() for x in ["GOOD", "EXCELLENT"]) else "rating-average"

#         return f"""
#         <div class="page-container">
#             <div class="header">
#                 <h1>{stock_data.get('Stock Name', 'N/A')}</h1>
#                 <div class="stock-info">{symbol} | {datetime.now().strftime('%d-%m-%Y')}</div>
#             </div>

#             <div class="main-price-bar">
#                 <span class="main-price">₹{stock_data.get('Price', 'N/A')}</span>
#                 <span class="pct-chg">({stock_data.get('% Chg', 'N/A')})</span>
#                 <span class="vol-text">Vol: {stock_data.get('Volume', 'N/A')}</span>
#             </div>

#             <div class="section-box">
#                 <div class="section-header">TECHNICAL ANALYSIS</div>
#                 {price_html}
#                 {indicators_html}
#             </div>

#             <div class="section-box">
#                 <div class="section-header">FUNDAMENTAL ANALYSIS</div>
#                 <div style="margin-bottom: 5px;">
#                     <span class="rating-badge {rating_class}">{rating_text}</span>
#                     <span class="risk-label">Risk: <strong>{fundamental.get('risk_level', 'N/A')}</strong></span>
#                 </div>
#                 <div class="scores-grid">{scores_html}</div>
#             </div>

#             <div class="chart-container">
#                 <div class="section-header">PRICE CHART</div>
#                 <div class="crop-viewport">
#                     <img src="file:///{abs_screenshot_path}" class="chart-img" alt="Chart Missing">
#                 </div>
#             </div>
#         </div>
#         """

#     all_stocks_html = "".join([generate_stock_html(i+1, row) for i, row in df.iterrows()])

#     complete_html = f"""
# <!DOCTYPE html>
# <html>
# <head>
#     <style>
#         @page {{ size: A4; margin: 0; }}
#         body {{ font-family: 'Helvetica', sans-serif; margin: 0; padding: 0; background: #fff; color: #222; }}

#         .page-container {{
#             width: 210mm;
#             height: 296mm; /* Strictly A4 height */
#             padding: 10mm;
#             box-sizing: border-box;
#             page-break-after: always;
#             overflow: hidden;
#             display: flex;
#             flex-direction: column;
#         }}

#         .header {{ background: #1a73e8; color: white; padding: 10px; text-align: center; border-radius: 4px; }}
#         .header h1 {{ margin: 0; font-size: 18px; }}

#         .main-price-bar {{ text-align: center; padding: 10px 0; border-bottom: 1px solid #eee; }}
#         .main-price {{ font-size: 24px; font-weight: bold; }}
#         .pct-chg {{ font-size: 18px; color: #28a745; margin-left: 10px; }}
#         .vol-text {{ color: #666; font-size: 12px; margin-left: 15px; }}

#         .section-box {{ margin-top: 10px; }}
#         .section-header {{ font-size: 12px; font-weight: bold; color: #1a73e8; border-left: 4px solid #1a73e8; padding-left: 8px; margin-bottom: 8px; }}

#         .price-grid {{ display: flex; justify-content: space-between; margin-bottom: 8px; background: #f8f9fa; padding: 5px; border-radius: 4px; }}
#         .price-item {{ text-align: center; flex: 1; }}
#         .price-label {{ font-size: 9px; color: #777; display: block; }}
#         .price-value {{ font-size: 11px; font-weight: bold; }}

#         .indicators-grid, .scores-grid {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 4px; }}
#         .scores-grid {{ grid-template-columns: repeat(3, 1fr); }}

#         .tech-item, .score-card {{ border: 1px solid #eee; padding: 4px; text-align: center; border-radius: 3px; }}
#         .label, .name {{ font-size: 8px; color: #888; display: block; }}
#         .value, .score {{ font-size: 10px; font-weight: bold; }}

#         .rating-badge {{ padding: 2px 8px; border-radius: 10px; color: white; font-size: 10px; font-weight: bold; }}
#         .rating-good {{ background: #28a745; }}
#         .rating-average {{ background: #ffc107; color: #000; }}

#         /* Chart Cropping Logic */
#         .chart-container {{ flex-grow: 1; margin-top: 10px; min-height: 0; }}
#         .crop-viewport {{
#             width: 100%;
#             height: 420px; /* Adjusted to take remaining space on A4 */
#             overflow: hidden;
#             border: 1px solid #ddd;
#             position: relative;
#             background: #fdfdfd;
#         }}
#         .chart-img {{
#             width: 100%;
#             position: absolute;
#             top: -50px; /* Adjust this to hide the top website menu */
#             left: 0;
#         }}

#         .overbought {{ color: #dc3545; }}
#         .oversold {{ color: #28a745; }}

#         .summary-container {{
#             width: 90%;
#             border: 2px solid #1a73e8;
#             border-radius: 12px;
#             padding: 30px;
#             background: white;
#         }}

#         .summary-header {{
#             display: flex;
#             align-items: center;
#             gap: 15px;
#             border-bottom: 1px solid #eee;
#             padding-bottom: 15px;
#             margin-bottom: 20px;
#         }}
#     </style>
# </head>
# <body>
#     {all_stocks_html}

#     <div class="summary-container">
#         <div class="summary-header">
#             <span class="icon">⚡</span>
#             <h2>TECHNICALLY STRONG STOCKS</h2>
#            <div> {filtered_stocks} </div>
#         </div>
#     </div>
#     <div class="summary-container">
#         <div class="summary-header">
#             <span class="icon">⚡</span>
#             <h2>TECHNICALLY AND FUNDAMENTALLY STRONG STOCKS</h2>
#            <div> {best_stock_ticker} </div>
#         </div>
#     </div>

# </body>
# </html>
# """
#     output_html_path = os.path.join(today_folder, "stock_analysis_report.html")
#     with open(output_html_path, "w", encoding="utf-8") as f:
#         f.write(complete_html)
#     print(f"Success! Saved to {output_html_path}")

Success! Saved to 03-01-2026/stock_analysis_report.html


In [ ]:
# from playwright.async_api import async_playwright
# import asyncio
# import os

# async def convert_html_to_pdf(html_file_path, output_pdf_path):
#     async with async_playwright() as p:
#         # Launch headless browser
#         browser = await p.chromium.launch()
#         page = await browser.new_page()

#         # Load the local HTML file
#         # Use an absolute path for the file URL
#         absolute_html_path = os.path.abspath(html_file_path)
#         await page.goto(f"file:///{absolute_html_path}")

#         # Wait for images to load before printing
#         await page.wait_for_load_state("networkidle")

#         # Emulate print media for A4 size
#         await page.pdf(
#             path=output_pdf_path,
#             format="A4",
#             print_background=True, # Critical for CSS colors and backgrounds
#             margin={"top": "1cm", "right": "1cm", "bottom": "1cm", "left": "1cm"},
#             display_header_footer=False
#         )

#         await browser.close()
#         print(f"PDF successfully created at: {output_pdf_path}")

# # --- Execution ---
# html_path = os.path.join(today_folder, "stock_analysis_report.html")
# pdf_path = os.path.join(today_folder, "Final_Stock_Analysis.pdf")

# # Generate the PDF
# await convert_html_to_pdf(html_path, pdf_path)

PDF successfully created at: 03-01-2026/Final_Stock_Analysis.pdf


In [ ]:
# image_paths = []

# today_folder = datetime.today().strftime('%d-%m-%Y') + "/" + "ScreenShot"
# for i in best_stock_ticker:
#     image_path = os.path.join(today_folder, f"{i}.png")
#     image_paths.append(image_path)

In [ ]:
# from IPython.core.display import Image
# #SEND
# import os
# import smtplib
# from email.message import EmailMessage

# SMTP_HOST = "smtp.gmail.com"
# SMTP_PORT = 587

# # EMAIL = os.environ["MY_EMAIL"]          # your@gmail.com
# # APP_PASSWORD = os.environ["MY_APP_PASS"]  # app password

# EMAIL = "knpatel0707@gmail.com"
# APP_PASSWORD = "ules habb jtdm jvni"

# msg = EmailMessage()
# msg["Subject"] = f"Best Stocks for tommorow"
# msg["From"] = EMAIL
# msg["To"] = "kevalnpatel070@gmail.com"
# msg.set_content("Hello — todays best stocks!")

# # Check if complete_html is defined and not empty before adding it
# if 'complete_html' in locals() and complete_html:
#     msg.add_alternative(complete_html, subtype="html")
# else:
#     print("HTML content not available for email.")

# for img_path in image_paths:
#     with open(img_path, "rb") as f:
#         file_data = f.read()
#         file_name = img_path
#         # Guess type from file extension
#         subtype = img_path.lstrip(".").lower()
#         if subtype not in ["png", "jpg", "jpeg"]:
#             subtype = "octet-stream"
#         msg.add_attachment(file_data,
#                           maintype="image",
#                           subtype=subtype,
#                           filename=file_name)


# if 'complete_html' in locals() and complete_html:
#     with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as smtp:
#         smtp.ehlo()
#         smtp.starttls()
#         smtp.login(EMAIL, APP_PASSWORD)
#         smtp.send_message(msg)

#     print("Email sent successfully!")
# else:
#     print("Email not sent because HTML content was not available.")

Email sent successfully!


In [ ]:
import os
import smtplib
from email.message import EmailMessage

# 1. Setup paths (Matching your previous logic)
today_folder = datetime.today().strftime("%d-%m-%Y")
pdf_path = os.path.join(today_folder, "Final_Stock_Analysis.pdf")

# 2. Extract Tickers for Response
# Assuming df is your filtered dataframe from previous steps
best_stock_tickers = df['Symbol'].tolist()

# --- EMAIL SENDING LOGIC ---
SMTP_HOST = "smtp.gmail.com"
SMTP_PORT = 587
EMAIL = "knpatel0707@gmail.com"
APP_PASSWORD = "ules habb jtdm jvni"

msg = EmailMessage()
msg["Subject"] = f"Best Stocks for Tomorrow - {today_folder}"
msg["From"] = EMAIL
msg["To"] = "kevalnpatel070@gmail.com"

# Plain text body
body_content = f"""Hello,

Please find the attached PDF for today's best stock analysis.

Best Stock Tickers:
{', '.join(best_stock_tickers)}

Technically Strong Stocks:
{', '.join(filtered_stocks_list)}

Best regards,
Stock Scanner Bot"""

msg.set_content(body_content)

# 3. Attach the PDF file
if os.path.exists(pdf_path):
    with open(pdf_path, "rb") as f:
        file_data = f.read()
        file_name = os.path.basename(pdf_path)
        msg.add_attachment(
            file_data,
            maintype="application",
            subtype="pdf",
            filename=file_name
        )

    # Send the Email
    try:
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as smtp:
            smtp.ehlo()
            smtp.starttls()
            smtp.login(EMAIL, APP_PASSWORD)
            smtp.send_message(msg)
        print(f"Email sent successfully with attachment: {file_name}")
    except Exception as e:
        print(f"Failed to send email: {e}")
else:
    print(f"Error: PDF file not found at {pdf_path}. Email not sent.")

# 4. Response Output
print("\n--- Summary ---")
print(f"Best Stock Tickers: {best_stock_tickers}")
print(f"Filtered (Technically Strong): {filtered_stocks_list}")

Email sent successfully with attachment: Final_Stock_Analysis.pdf

--- Summary ---
Best Stock Tickers: ['SANDHAR', 'CARRARO', 'ROLEXRINGS', 'AUTOAXLES', 'MANORAMA', 'VOLTAS', 'RELIABLE']
Filtered (Technically Strong): ['SANDHAR', 'VGL', 'CARRARO', 'SANSERA', 'ROLEXRINGS', 'SUPREMEENG', 'AUTOAXLES', 'UJJIVANSFB', 'MANORAMA', 'RHETAN', 'VOLTAS', 'RELIABLE']
